# E07 FINAL KAGGLE RUN
## FINAL HANDOFF VERSION

**RUN ON KAGGLE GPU.**  
**RUN_MODE is already set to FULL.**  
**Do not run locally on CPU.**  
**Do not change the scientific protocol, seeds, models, or evaluation split.**  
**Do not modify model code, seed list, dataset split, preprocessing, or training configuration.**  
**Expected experiment size: 25 training runs.**

---

# 1. E07 Final Multi-Seed Robustness Experiment
## TE-Q-Transformer V2 Research Project

**Scientific research question:**  
*"How sensitive are the observed E06 performance results to random initialization across a predefined set of five random seeds?"*

E07 is a **multi-seed reproducibility / robustness** experiment. It reuses the **exact E06 generalization protocol** and the **five designated models** from `baselineComparison.ipynb`. It does **not** establish universal statistical proof, statistical significance, unseen-temperature generalization, or quantum advantage.

---

### Designated models (exactly five)
1. **TE-Q-Transformer** (proposed physics-guided quantum Transformer; 92,554 parameters)
2. **QNN-GRU** (hybrid QNN + GRU; 29,533 parameters)
3. **iTransformer** (inverted variate token Transformer; 137,473 parameters)
4. **Transformer** (classical multi-head attention encoder; 80,257 parameters)
5. **PatchTST** (channel-independent patch Transformer; 109,962 parameters)

### Fixed seed set
`SEEDS = [42, 43, 44, 45, 46]`

Seed 42 is included because E01–E06 used seed 42. The seed set is predefined and must not be altered based on results.

**Total planned runs:** 5 models × 5 seeds = **25 sequential training runs**.

### Evaluation protocol (identical to E06)
- **Train:** B0005, B0006, B0007, B0029, B0030, B0031, and B0053 cycles 0–36 (660 cycles)
- **Test / unseen cells:** B0018 (n = 132), B0032 (n = 39)
- **Test / temporal extrapolation:** B0053 cycles 37–52 (n = 16). B0053 is **not** an unseen cell.
- E07 is **not** an unseen-temperature experiment.

### How to run on Kaggle
1. Select GPU accelerator (P100 or T4).
2. Attach the NASA `.npy` dataset.
3. **Do not change `RUN_MODE`.** It is already `FULL`.
4. Run All.
5. Wait for all 25 runs. If the session disconnects, Run All again; verified complete runs are skipped.
6. Download `E07_results/`.


## 2. Environment


In [ ]:
# ==============================================================================
# SECTION 1: ENVIRONMENT
# ==============================================================================
import os
import sys
import gc
import time
import math
import json
import random
import platform
import subprocess
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

try:
    import pennylane as qml
except ImportError:
    print("[Setup] PennyLane not detected. Installing via pip...")
    os.system(f"{sys.executable} -m pip install -q pennylane")
    import pennylane as qml

import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 75)
print("E07 ENVIRONMENT")
print("=" * 75)
print(f"Python:           {platform.python_version()}")
print(f"PyTorch:          {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
print(f"CUDA version:     {torch.version.cuda}")
print(f"device:           {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU name:         {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory (GB):  {props.total_memory / 1e9:.2f}")
print(f"PennyLane:        {qml.__version__}")
print("=" * 75)


## 3. Configuration

All output paths are derived from `OUTPUT_ROOT`. Dataset discovery writes `DATA_ROOT`. Override `DATA_ROOT_OVERRIDE` only if the NASA `.npy` files live in a non-standard folder. Do not hard-code local Windows drive letters.

`RUN_MODE` is **FULL**. `DRY_RUN` remains available only as a secondary debug option and must not be used for the Lisan handoff.


In [ ]:
# ==============================================================================
# SECTION 2: CONFIGURATION
# ==============================================================================
# "FULL"    = 5 models x 5 seeds = 25 training runs (Kaggle GPU). DEFAULT FOR HANDOFF.
# "DRY_RUN" = 1-epoch / 1-batch pipeline check only. Do NOT use for the Lisan GPU run.
RUN_MODE = "FULL"

SEEDS = [42, 43, 44, 45, 46]
assert SEEDS == [42, 43, 44, 45, 46], "E07 seed set is fixed and must not be changed."
assert len(SEEDS) == 5 and len(set(SEEDS)) == 5, "E07 requires five unique seeds."

EXECUTION_ORDER = [
    "TE-Q-Transformer",
    "QNN-GRU",
    "iTransformer",
    "Transformer",
    "PatchTST",
]

# Optional explicit data directory. Leave None to auto-discover B0005_X.npy.
DATA_ROOT_OVERRIDE = None  # e.g. Path("/kaggle/input/your-nasa-dataset")
OUTPUT_ROOT = Path("E07_results")

CHECKPOINTS_DIR = OUTPUT_ROOT / "checkpoints"
METRICS_DIR = OUTPUT_ROOT / "metrics"
PREDICTIONS_DIR = OUTPUT_ROOT / "predictions"
TRAINING_DIR = OUTPUT_ROOT / "training"
CONFIGS_DIR = OUTPUT_ROOT / "configs"
REPORTS_DIR = OUTPUT_ROOT / "reports"
PROVENANCE_DIR = OUTPUT_ROOT / "provenance"

for d in [CHECKPOINTS_DIR, METRICS_DIR, PREDICTIONS_DIR, TRAINING_DIR, CONFIGS_DIR, REPORTS_DIR, PROVENANCE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NASA_FULL_TRAIN_CELLS = ("B0005", "B0006", "B0007", "B0029", "B0030", "B0031")
NASA_FULL_TEST_CELLS = ("B0018", "B0032")
NASA_SPLIT_CELL_ID = "B0053"
NASA_SPLIT_RATIO = 0.70
FEATURE_IDX_TO_SCALE = (0, 1, 3)  # V, I, Time_norm; Temperature_C stays raw Celsius
SEQUENCE_LENGTH = 512
FEATURES = ["V", "I", "T_C", "Time_norm"]
N_TRAIN_EXPECTED = 660
N_TEST_EXPECTED = {"B0018": 132, "B0032": 39, "B0053_test": 16}

print(f"RUN_MODE     = {RUN_MODE}")
print("E07 SEEDS:")
print(SEEDS)
print(f"OUTPUT_ROOT  = {OUTPUT_ROOT.resolve()}")
print(f"DATA_ROOT_OVERRIDE = {DATA_ROOT_OVERRIDE}")
print(f"Planned runs = {len(EXECUTION_ORDER) * len(SEEDS)}")
print("TE-Q weight_decay=0.01 is retained from the established E01/E05/E06 implementation/campaign and is not tuned during E07.")
print("QNN-GRU batch_size=16 is retained from the E05/E06 protocol and is not an E07 retune.")

if RUN_MODE == "FULL" and not torch.cuda.is_available():
    raise RuntimeError(
        "E07 FULL mode requires a CUDA GPU. "
        "Please run the notebook on Kaggle with GPU enabled."
    )
if not torch.cuda.is_available():
    print("WARNING: CUDA is unavailable. FULL E07 must not be run on CPU.")


## 4. Dataset Discovery


In [ ]:
# ==============================================================================
# SECTION 3: DATASET DISCOVERY
# ==============================================================================
def discover_nasa_data_dir(override: Optional[Path] = None) -> Path:
    if override is not None:
        override = Path(override)
        if not (override / "B0005_X.npy").exists():
            raise FileNotFoundError(f"DATA_ROOT_OVERRIDE has no B0005_X.npy: {override}")
        return override

    candidates = [
        Path("Dataset/nasa"),
        Path("../Dataset/nasa"),
        Path("data/processed/nasa_randomized"),
        Path("../input/nasa-battery-dataset/data/processed/nasa_randomized"),
        Path("../input/nasa-battery-dataset"),
        Path("../input/nasa-battery-data"),
        Path("../input/nasa_randomized"),
        Path("/kaggle/input/nasa-battery-dataset/data/processed/nasa_randomized"),
        Path("/kaggle/input/nasa-battery-dataset"),
        Path("/kaggle/input/nasa-battery-data"),
        Path("/kaggle/input"),
    ]
    for p in candidates:
        if p.exists() and (p / "B0005_X.npy").exists():
            return p
        if p.exists() and p.name == "input":
            for found in p.rglob("B0005_X.npy"):
                return found.parent

    for search_root in [Path("."), Path(".."), Path("/kaggle/input")]:
        if search_root.exists():
            for found in search_root.rglob("B0005_X.npy"):
                return found.parent
    raise FileNotFoundError(
        "Could not find NASA B0005_X.npy. Attach the NASA .npy dataset or set DATA_ROOT_OVERRIDE."
    )

DATA_ROOT = discover_nasa_data_dir(DATA_ROOT_OVERRIDE)
print(f"[Data Discovery] DATA_ROOT = {DATA_ROOT.resolve()}")

REQUIRED_CELLS = ["B0005", "B0006", "B0007", "B0018", "B0029", "B0030", "B0031", "B0032", "B0053"]
print("=" * 75)
print(f"{'Cell':10s} | {'X Shape':18s} | {'SOH Shape':12s} | {'y[0]':10s}")
print("=" * 75)
for cell_id in REQUIRED_CELLS:
    x_file = DATA_ROOT / f"{cell_id}_X.npy"
    y_file = DATA_ROOT / f"{cell_id}_soh.npy"
    if not x_file.exists() or not y_file.exists():
        raise FileNotFoundError(f"Missing {cell_id} arrays in {DATA_ROOT}")
    x_arr = np.load(x_file)
    y_arr = np.load(y_file)
    print(f"{cell_id:10s} | {str(tuple(x_arr.shape)):18s} | {str(tuple(y_arr.shape)):12s} | {float(y_arr[0]):.4f}")
print("=" * 75)


## 5. Preprocessing

MinMax is fitted on the 660 training cycles only. Voltage, Current, and Time_norm are scaled. Temperature remains raw Celsius. Test cells and the B0053 holdout are never used to fit the scaler.


In [ ]:
# ==============================================================================
# SECTION 4: PREPROCESSING (IDENTICAL TO E06)
# ==============================================================================
class NASABatteryDataset(Dataset):
    def __init__(self, X: torch.Tensor, y: torch.Tensor, cycle_ids=None) -> None:
        self.X = X.float()
        self.y = y.float()
        if cycle_ids is None:
            self.cycle_ids = torch.arange(len(self.y), dtype=torch.long)
        else:
            self.cycle_ids = torch.as_tensor(cycle_ids, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, int]:
        return self.X[idx], self.y[idx], int(self.cycle_ids[idx])


def load_cell_arrays(data_dir: Path, cell_id: str) -> Tuple[np.ndarray, np.ndarray]:
    X = np.load(data_dir / f"{cell_id}_X.npy").astype(np.float32, copy=True)
    y = np.load(data_dir / f"{cell_id}_soh.npy").astype(np.float32, copy=False)
    return X, y


def normalize_soh_per_cell(y: np.ndarray) -> np.ndarray:
    c0 = float(y[0])
    return (y / np.float32(c0)).astype(np.float32, copy=False)


def load_full_cell(data_dir: Path, cell_id: str) -> Tuple[torch.Tensor, torch.Tensor]:
    X, y = load_cell_arrays(data_dir, cell_id)
    y = normalize_soh_per_cell(y)
    return torch.from_numpy(X).float(), torch.from_numpy(y).float()


def split_cell_70_30(data_dir: Path, cell_id: str):
    X, y = load_cell_arrays(data_dir, cell_id)
    y = normalize_soh_per_cell(y)
    split_idx = int(round(len(X) * NASA_SPLIT_RATIO))
    train_X = torch.from_numpy(X[:split_idx]).float()
    train_y = torch.from_numpy(y[:split_idx]).float()
    test_X = torch.from_numpy(X[split_idx:]).float()
    test_y = torch.from_numpy(y[split_idx:]).float()
    return (train_X, train_y, split_idx), (test_X, test_y, split_idx)


def build_feature_scaler(train_X_tensors: List[torch.Tensor]) -> MinMaxScaler:
    train_concat = torch.cat(train_X_tensors, dim=0).numpy()
    B_total, L, D = train_concat.shape
    train_reshaped = train_concat.reshape(-1, D)
    scaler = MinMaxScaler()
    scaler.fit(train_reshaped[:, list(FEATURE_IDX_TO_SCALE)])
    return scaler


def apply_feature_scaler(X: torch.Tensor, scaler: MinMaxScaler) -> torch.Tensor:
    X_np = X.numpy() if isinstance(X, torch.Tensor) else np.asarray(X)
    B, L, D = X_np.shape
    X_reshaped = X_np.reshape(-1, D).copy()
    X_reshaped[:, list(FEATURE_IDX_TO_SCALE)] = scaler.transform(X_reshaped[:, list(FEATURE_IDX_TO_SCALE)])
    return torch.from_numpy(X_reshaped.reshape(B, L, D)).float()


def compute_metrics(actual: np.ndarray, predicted: np.ndarray) -> Dict[str, float]:
    actual = np.asarray(actual, dtype=np.float64).reshape(-1)
    predicted = np.asarray(predicted, dtype=np.float64).reshape(-1)
    abs_error = np.abs(actual - predicted)
    denom = np.clip(np.abs(actual), a_min=1e-8, a_max=None)
    rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
    mae = float(mean_absolute_error(actual, predicted))
    mape = float(np.mean(abs_error / denom) * 100.0)
    max_e = float(np.max(abs_error))
    if np.allclose(actual, actual[0]):
        r2 = float("nan")
    else:
        r2 = float(r2_score(actual, predicted))
    return {"RMSE": rmse, "MAE": mae, "MAPE (%)": mape, "R2": r2, "MaxE": max_e}


def get_nasa_dataloaders(
    data_dir: Path,
    batch_size: int = 8,
    seed: int = 42,
    num_workers: int = 0,
):
    train_X_list, train_y_list = [], []
    for cell_id in NASA_FULL_TRAIN_CELLS:
        X, y = load_full_cell(data_dir, cell_id)
        train_X_list.append(X)
        train_y_list.append(y)

    (split_tr_X, split_tr_y, split_idx), (split_te_X, split_te_y, _) = split_cell_70_30(data_dir, NASA_SPLIT_CELL_ID)
    train_X_list.append(split_tr_X)
    train_y_list.append(split_tr_y)

    # Leakage guards (must hold before every training run)
    assert "B0018" not in NASA_FULL_TRAIN_CELLS
    assert "B0032" not in NASA_FULL_TRAIN_CELLS
    assert set(NASA_FULL_TRAIN_CELLS).isdisjoint(set(NASA_FULL_TEST_CELLS))
    assert split_idx == 37, f"B0053 train split_idx must be 37 (cycles 0-36), got {split_idx}"
    assert len(split_te_y) == 16, f"B0053 test must be 16 cycles (37-52), got {len(split_te_y)}"

    scaler = build_feature_scaler(train_X_list)
    expected_scaler_n = N_TRAIN_EXPECTED * SEQUENCE_LENGTH
    assert int(scaler.n_samples_seen_) == expected_scaler_n, (
        f"Scaler saw {scaler.n_samples_seen_} points, expected {expected_scaler_n}"
    )

    scaled_train_X = torch.cat([apply_feature_scaler(x, scaler) for x in train_X_list], dim=0)
    scaled_train_y = torch.cat(train_y_list, dim=0)
    assert len(scaled_train_y) == N_TRAIN_EXPECTED

    g = torch.Generator()
    g.manual_seed(int(seed))
    train_loader = DataLoader(
        NASABatteryDataset(scaled_train_X, scaled_train_y),
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
        generator=g,
        pin_memory=torch.cuda.is_available(),
    )

    test_loaders: Dict[str, DataLoader] = {}
    for cell_id in NASA_FULL_TEST_CELLS:
        raw_X, raw_y = load_full_cell(data_dir, cell_id)
        sc_X = apply_feature_scaler(raw_X, scaler)
        cyc = np.arange(len(raw_y), dtype=np.int64)
        test_loaders[cell_id] = DataLoader(
            NASABatteryDataset(sc_X, raw_y, cyc),
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    sc_b53 = apply_feature_scaler(split_te_X, scaler)
    cyc53 = np.arange(split_idx, split_idx + len(split_te_y), dtype=np.int64)
    test_loaders["B0053_test"] = DataLoader(
        NASABatteryDataset(sc_b53, split_te_y, cyc53),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    for k, n_exp in N_TEST_EXPECTED.items():
        assert len(test_loaders[k].dataset) == n_exp, f"{k} n={len(test_loaders[k].dataset)} expected {n_exp}"

    return train_loader, test_loaders, scaler, split_idx


## 6. Split Audit

B0018 and B0032 are unseen cells. B0053 final 30% is temporal extrapolation of a seen cell. This is not an unseen-temperature protocol.


In [ ]:
# ==============================================================================
# SECTION 5: EXACT E07 SPLIT AUDIT
# ==============================================================================
print("=" * 85)
print(f"{'Cell':10s} | {'Train/Test':12s} | {'Evaluation Type':24s} | {'Cycles':30s}")
print("=" * 85)
print(f"{'B0005':10s} | {'TRAIN':12s} | {'training pool':24s} | {'all cycles':30s}")
print(f"{'B0006':10s} | {'TRAIN':12s} | {'training pool':24s} | {'all cycles':30s}")
print(f"{'B0007':10s} | {'TRAIN':12s} | {'training pool':24s} | {'all cycles':30s}")
print(f"{'B0029':10s} | {'TRAIN':12s} | {'training pool':24s} | {'all cycles':30s}")
print(f"{'B0030':10s} | {'TRAIN':12s} | {'training pool':24s} | {'all cycles':30s}")
print(f"{'B0031':10s} | {'TRAIN':12s} | {'training pool':24s} | {'all cycles':30s}")
print(f"{'B0053':10s} | {'TRAIN':12s} | {'seen-cell prefix':24s} | {'cycles 0-36 (37 cycles)':30s}")
print(f"{'B0018':10s} | {'TEST':12s} | {'unseen_cell':24s} | {'all cycles (132)':30s}")
print(f"{'B0032':10s} | {'TEST':12s} | {'unseen_cell':24s} | {'all cycles (39)':30s}")
print(f"{'B0053':10s} | {'TEST':12s} | {'temporal_extrapolation':24s} | {'cycles 37-52 (16)':30s}")
print("=" * 85)

_audit_loader, _audit_test, _audit_scaler, _split_idx = get_nasa_dataloaders(DATA_ROOT, batch_size=8, seed=42)
assert len(_audit_loader.dataset) == 660
assert _split_idx == 37
assert len(_audit_test["B0018"].dataset) == 132
assert len(_audit_test["B0032"].dataset) == 39
assert len(_audit_test["B0053_test"].dataset) == 16
print("[Split audit] PASS: 660 train / 187 test; B0018 unseen; B0032 unseen; B0053 70/30 temporal.")
print("[Scaler audit] PASS: MinMax fitted on training cycles only; T_C unscaled.")
del _audit_loader, _audit_test, _audit_scaler
gc.collect()


## 7. Model Registry

Architectures below are copied from `baselineComparison.ipynb` cell 4 (baselines) and `src/models/teq_transformer.py` (TE-Q). They are not rewritten. Only the five designated models are registered and trained.


In [ ]:
# ==============================================================================
# SECTION 6: ARCHITECTURAL DEFINITIONS (CANONICAL, UNCHANGED)
# PART A: baselineComparison.ipynb cell 4 (all 10 baseline classes; only 5 are registered)
# PART B: src/models/teq_transformer.py
# ==============================================================================

# ==============================================================================
# 4. ARCHITECTURAL DEFINITIONS FOR ALL 10 ACTIVE BASELINE MODELS
# (100% Faithful Line-by-Line Reproduction of Repository Baseline Suite)
# ==============================================================================

# ------------------------------------------------------------------------------
# Model 1: LSTM (Hochreiter & Schmidhuber, 1997)
# ------------------------------------------------------------------------------
"""Standard LSTM baseline model for battery SOH estimation."""


import torch
from torch import nn


class LSTMModel(nn.Module):
    """LSTM sequence model with input projection and regression head."""

    def __init__(
        self,
        input_dim: int = 4,
        d_model: int = 64,
        num_layers: int = 2,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
        use_projection: bool = True,
        pooling: str = "last",
    ) -> None:
        super().__init__()
        self.pooling = pooling
        self.input_projection = nn.Linear(input_dim, d_model) if use_projection else nn.Identity()
        lstm_in = d_model if use_projection else input_dim
        self.lstm = nn.LSTM(
            input_size=lstm_in,
            hidden_size=d_model,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(d_model, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, 4]
        x = self.input_projection(x)
        outputs, (h_n, _) = self.lstm(x)
        if self.pooling == "mean":
            pooled = outputs.mean(dim=1)
        else:
            pooled = outputs[:, -1, :]
        return self.head(pooled).squeeze(-1)

# ------------------------------------------------------------------------------
# Model 2: GRU (Cho et al., 2014)
# ------------------------------------------------------------------------------
"""Standard GRU baseline model for battery SOH estimation."""


import torch
from torch import nn


class GRUModel(nn.Module):
    """GRU sequence model with input projection and regression head."""

    def __init__(
        self,
        input_dim: int = 4,
        d_model: int = 64,
        num_layers: int = 2,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
        use_projection: bool = True,
        pooling: str = "last",
    ) -> None:
        super().__init__()
        self.pooling = pooling
        self.input_projection = nn.Linear(input_dim, d_model) if use_projection else nn.Identity()
        gru_in = d_model if use_projection else input_dim
        self.gru = nn.GRU(
            input_size=gru_in,
            hidden_size=d_model,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(d_model, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_projection(x)
        outputs, hidden = self.gru(x)
        if self.pooling == "mean":
            pooled = outputs.mean(dim=1)
        else:
            pooled = outputs[:, -1, :]
        return self.head(pooled).squeeze(-1)

# ------------------------------------------------------------------------------
# Model 3: CNN1D (Kiranyaz et al., 2021)
# ------------------------------------------------------------------------------
"""1D Temporal Convolutional baseline model for battery SOH estimation."""


import torch
from torch import nn


class CNN1DModel(nn.Module):
    """1D CNN sequence model with multi-scale temporal convolutions and global pooling."""

    def __init__(
        self,
        input_dim: int = 4,
        d_model: int = 64,
        num_layers: int = 3,
        kernel_size: int = 5,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
    ) -> None:
        super().__init__()
        layers = []
        in_ch = input_dim
        for i in range(num_layers):
            out_ch = d_model
            layers.append(
                nn.Conv1d(
                    in_channels=in_ch,
                    out_channels=out_ch,
                    kernel_size=kernel_size,
                    padding=kernel_size // 2,
                )
            )
            layers.append(nn.BatchNorm1d(out_ch))
            layers.append(nn.GELU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_ch = out_ch

        self.conv_net = nn.Sequential(*layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, 4] -> permute to [B, 4, L]
        x_conv = x.transpose(1, 2)
        features = self.conv_net(x_conv)  # [B, d_model, L]
        pooled = features.mean(dim=-1)     # Global average pooling -> [B, d_model]
        return self.head(pooled).squeeze(-1)

# ------------------------------------------------------------------------------
# Model 4: TCN (Bai, Kolter, & Koltun, 2018)
# ------------------------------------------------------------------------------
"""Temporal Convolutional Network (TCN) baseline model for battery SOH estimation."""


import torch
from torch import nn


class ChausalDilatedConv1DBlock(nn.Module):
    """Causal dilated conv block with residual connection."""

    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, dilation: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation, padding=self.padding)
        self.act1 = nn.GELU()
        self.drop1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, dilation=dilation, padding=self.padding)
        self.act2 = nn.GELU()
        self.drop2 = nn.Dropout(dropout)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Causal trim: drop trailing padding
        res = self.residual(x)
        out = self.conv1(x)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        out = self.drop1(self.act1(out))
        out = self.conv2(out)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        out = self.drop2(self.act2(out))
        return out + res


class TCNModel(nn.Module):
    """Deep Temporal Convolutional Network with exponential dilations."""

    def __init__(
        self,
        input_dim: int = 4,
        d_model: int = 64,
        kernel_size: int = 3,
        num_levels: int = 4,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
    ) -> None:
        super().__init__()
        layers = []
        in_ch = input_dim
        for i in range(num_levels):
            dilation = 2 ** i
            layers.append(
                ChausalDilatedConv1DBlock(
                    in_channels=in_ch,
                    out_channels=d_model,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout,
                )
            )
            in_ch = d_model

        self.network = nn.Sequential(*layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, 4] -> [B, 4, L]
        x_in = x.transpose(1, 2)
        feat = self.network(x_in)
        pooled = feat[:, :, -1]  # Last causal step
        return self.head(pooled).squeeze(-1)

# ------------------------------------------------------------------------------
# Model 5: DLinear (Zeng et al., AAAI 2023)
# ------------------------------------------------------------------------------
"""DLinear baseline (LTSF-Linear family) adapted for NASA SOH seq-to-one regression.

Provenance (official upstream):
- Paper: "Are Transformers Effective for Time Series Forecasting?" (arXiv:2205.13504; AAAI 2023 per repo)
- Official repo: https://github.com/cure-lab/LTSF-Linear/
- Upstream file: models/DLinear.py
- Upstream commit (HEAD verified 2026-09-15): 0c113668a3b88c4c4ee586b8c5ec3e539c4de5a6
- License: Apache-2.0 (https://github.com/cure-lab/LTSF-Linear/blob/main/LICENSE)

Core architecture preserved? YES (series decomposition + linear seasonal/trend heads).
Task adaptation:
- Set pred_len = 1 (single-step output) and map the resulting channel vector to a scalar SOH via a small linear head.
"""


from dataclasses import dataclass

import torch
from torch import nn


# --------------------------------------------------------------------------------------
# Minimal upstream-derived core (Apache-2.0):
# This code is adapted from cure-lab/LTSF-Linear/models/DLinear.py with minimal changes.
# --------------------------------------------------------------------------------------


class _MovingAvg(nn.Module):
    """Moving average block to highlight the trend of time series (upstream: moving_avg)."""

    def __init__(self, kernel_size: int, stride: int = 1) -> None:
        super().__init__()
        self.kernel_size = int(kernel_size)
        self.avg = nn.AvgPool1d(kernel_size=self.kernel_size, stride=stride, padding=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, C]
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1)
        x_pad = torch.cat([front, x, end], dim=1)  # [B, L + pad, C]
        x_avg = self.avg(x_pad.permute(0, 2, 1)).permute(0, 2, 1)  # [B, L, C]
        return x_avg


class _SeriesDecomp(nn.Module):
    """Series decomposition block (upstream: series_decomp)."""

    def __init__(self, kernel_size: int) -> None:
        super().__init__()
        self.moving_avg = _MovingAvg(kernel_size, stride=1)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        moving_mean = self.moving_avg(x)
        res = x - moving_mean
        return res, moving_mean


@dataclass(frozen=True)
class DLinearConfig:
    seq_len: int = 512
    pred_len: int = 1
    enc_in: int = 4
    individual: bool = False
    kernel_size: int = 25  # upstream default


class _DLinearCore(nn.Module):
    """Upstream DLinear forward: [B, seq_len, C] -> [B, pred_len, C]."""

    def __init__(self, cfg: DLinearConfig) -> None:
        super().__init__()
        self.seq_len = cfg.seq_len
        self.pred_len = cfg.pred_len
        self.channels = cfg.enc_in
        self.individual = cfg.individual

        self.decomposition = _SeriesDecomp(cfg.kernel_size)

        if self.individual:
            self.Linear_Seasonal = nn.ModuleList([nn.Linear(self.seq_len, self.pred_len) for _ in range(self.channels)])
            self.Linear_Trend = nn.ModuleList([nn.Linear(self.seq_len, self.pred_len) for _ in range(self.channels)])
        else:
            self.Linear_Seasonal = nn.Linear(self.seq_len, self.pred_len)
            self.Linear_Trend = nn.Linear(self.seq_len, self.pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, C]
        seasonal_init, trend_init = self.decomposition(x)
        seasonal_init = seasonal_init.permute(0, 2, 1)  # [B, C, L]
        trend_init = trend_init.permute(0, 2, 1)        # [B, C, L]

        if self.individual:
            seasonal_output = torch.zeros(
                (seasonal_init.size(0), seasonal_init.size(1), self.pred_len),
                dtype=seasonal_init.dtype,
                device=seasonal_init.device,
            )
            trend_output = torch.zeros_like(seasonal_output)
            for i in range(self.channels):
                seasonal_output[:, i, :] = self.Linear_Seasonal[i](seasonal_init[:, i, :])
                trend_output[:, i, :] = self.Linear_Trend[i](trend_init[:, i, :])
        else:
            seasonal_output = self.Linear_Seasonal(seasonal_init)  # [B, C, pred_len]
            trend_output = self.Linear_Trend(trend_init)          # [B, C, pred_len]

        out = seasonal_output + trend_output  # [B, C, pred_len]
        return out.permute(0, 2, 1)  # [B, pred_len, C]


class DLinearSOHModel(nn.Module):
    """DLinear adapted to SOH regression: [B, 512, 4] -> [B]."""

    def __init__(
        self,
        seq_len: int = 512,
        enc_in: int = 4,
        kernel_size: int = 25,
        individual: bool = False,
        head: str = "linear",  # how to map channel vector -> scalar
    ) -> None:
        super().__init__()
        cfg = DLinearConfig(seq_len=seq_len, pred_len=1, enc_in=enc_in, individual=individual, kernel_size=kernel_size)
        self.core = _DLinearCore(cfg)

        if head == "mean":
            self.scalar_head = None
            self.head_mode = "mean"
        elif head == "linear":
            self.scalar_head = nn.Linear(enc_in, 1)
            self.head_mode = "linear"
        else:
            raise ValueError(f"Unknown head='{head}'. Use 'linear' or 'mean'.")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3 or x.shape[1] != 512 or x.shape[2] != 4:
            raise ValueError(f"Expected [B, 512, 4], got {tuple(x.shape)}")
        y_seq = self.core(x)              # [B, 1, 4]
        y_vec = y_seq[:, 0, :]            # [B, 4]
        if self.head_mode == "mean":
            y = y_vec.mean(dim=1, keepdim=True)
        else:
            y = self.scalar_head(y_vec)   # [B, 1]
        return y.squeeze(-1)


__all__ = ["DLinearSOHModel"]

# ------------------------------------------------------------------------------
# Model 6: Classical Transformer (Vaswani et al., 2017)
# ------------------------------------------------------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 1024) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]

"""Classical Transformer Encoder baseline (matching nasa-te-q-transformer-transformer.ipynb)."""


import torch
from torch import nn


class TransformerModel(nn.Module):
    """Pure classical Transformer encoder baseline without quantum embedding."""

    def __init__(
        self,
        input_dim: int = 4,
        d_model: int = 64,
        n_heads: int = 2,
        n_layers: int = 3,
        dim_feedforward: int = 64,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
        use_cls_token: bool = True,
        pooling: str = "cls",
    ) -> None:
        super().__init__()
        self.use_cls_token = use_cls_token
        self.pooling = pooling
        self.input_projection = nn.Linear(input_dim, d_model)

        if use_cls_token:
            self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
            nn.init.trunc_normal_(self.cls_token, std=0.02)
        else:
            self.cls_token = None

        self.positional_encoding = PositionalEncoding(d_model=d_model, max_len=1024)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, 4]
        h = self.input_projection(x)
        if self.use_cls_token and self.cls_token is not None:
            cls = self.cls_token.expand(x.size(0), -1, -1)
            h = torch.cat([cls, h], dim=1)
        h = self.positional_encoding(h)
        encoded = self.encoder(h)
        if self.use_cls_token and self.pooling == "cls":
            pooled = encoded[:, 0, :]
        else:
            pooled = encoded.mean(dim=1)
        return self.head(pooled).squeeze(-1)

# ------------------------------------------------------------------------------
# Model 7: PatchTST (Nie et al., ICLR 2023)
# ------------------------------------------------------------------------------
"""PatchTST baseline adapted for NASA SOH seq-to-one regression.

Provenance (official upstream reference):
- Paper: "A Time Series is Worth 64 Words: Long-term Forecasting with Transformers" (ICLR 2023; arXiv:2211.14730)
  - Paper URL: https://arxiv.org/abs/2211.14730
- Official repo: https://github.com/yuqinie98/PatchTST
- Upstream commit (HEAD verified 2026-09-15): 204c21efe0b39603ad6e2ca640ef5896646ab1a9
- License: Apache-2.0 (https://github.com/yuqinie98/PatchTST/blob/main/LICENSE)

Core architecture preserved? YES (patching + channel-independence + Transformer encoder).

Implementation note:
The official repo is a forecasting framework. Here we implement the **core PatchTST design**
(patching + channel-independence + Transformer encoder) and adapt only the task interface to
SOH regression:
- input: [B, 512, 4]
- output: scalar SOH [B]

Task adaptation (minimal):
- pred_len set to 1 (single-step output).
- per-channel outputs are fused to a scalar via a small linear head.
"""


from dataclasses import dataclass

import torch
from torch import nn


class _LearnablePositionalEncoding(nn.Module):
    """Learnable positional encoding (PatchTST uses learnable PE by default)."""

    def __init__(self, length: int, d_model: int) -> None:
        super().__init__()
        self.pe = nn.Parameter(torch.zeros(1, length, d_model))
        nn.init.trunc_normal_(self.pe, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]


class _RevIN(nn.Module):
    """Reversible Instance Normalization (RevIN), compact implementation."""

    def __init__(self, num_features: int, affine: bool = True, subtract_last: bool = False, eps: float = 1e-5) -> None:
        super().__init__()
        self.affine = affine
        self.subtract_last = subtract_last
        self.eps = eps
        if affine:
            self.gamma = nn.Parameter(torch.ones(1, 1, num_features))
            self.beta = nn.Parameter(torch.zeros(1, 1, num_features))
        else:
            self.gamma = None
            self.beta = None
        self._last = None
        self._mean = None
        self._stdev = None

    def norm(self, x: torch.Tensor) -> torch.Tensor:
        if self.subtract_last:
            self._last = x[:, -1:, :].detach()
            x = x - self._last
        self._mean = x.mean(dim=1, keepdim=True).detach()
        x = x - self._mean
        self._stdev = torch.sqrt(torch.var(x, dim=1, keepdim=True, unbiased=False) + self.eps).detach()
        x = x / self._stdev
        if self.affine:
            x = x * self.gamma + self.beta
        return x

    def denorm(self, x: torch.Tensor) -> torch.Tensor:
        if self.affine:
            x = (x - self.beta) / (self.gamma + self.eps)
        x = x * self._stdev + self._mean
        if self.subtract_last:
            x = x + self._last
        return x


@dataclass(frozen=True)
class PatchTSTConfig:
    seq_len: int = 512
    enc_in: int = 4
    patch_len: int = 16
    stride: int = 8
    d_model: int = 64
    n_heads: int = 2
    e_layers: int = 3
    d_ff: int = 128
    dropout: float = 0.0
    revin: bool = True
    affine: bool = True
    subtract_last: bool = False


class PatchTSTSOHModel(nn.Module):
    """PatchTST (channel-independent) adapted to SOH regression: [B, 512, 4] -> [B]."""

    def __init__(
        self,
        seq_len: int = 512,
        enc_in: int = 4,
        patch_len: int = 16,
        stride: int = 8,
        d_model: int = 64,
        n_heads: int = 2,
        e_layers: int = 3,
        d_ff: int = 128,
        dropout: float = 0.0,
        revin: bool = True,
        affine: bool = True,
        subtract_last: bool = False,
        head_hidden_dim: int = 64,
    ) -> None:
        super().__init__()
        self.cfg = PatchTSTConfig(
            seq_len=seq_len,
            enc_in=enc_in,
            patch_len=patch_len,
            stride=stride,
            d_model=d_model,
            n_heads=n_heads,
            e_layers=e_layers,
            d_ff=d_ff,
            dropout=dropout,
            revin=revin,
            affine=affine,
            subtract_last=subtract_last,
        )

        patch_num = int((seq_len - patch_len) / stride + 1)
        self.revin = _RevIN(enc_in, affine=affine, subtract_last=subtract_last) if revin else None

        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_enc = _LearnablePositionalEncoding(length=patch_num, d_model=d_model)
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=e_layers)

        # Per-channel head (pred_len = 1)
        self.channel_head = nn.Linear(d_model * patch_num, 1)

        # Channel fusion to scalar SOH
        self.scalar_head = nn.Sequential(
            nn.Linear(enc_in, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3 or x.shape[1] != self.cfg.seq_len or x.shape[2] != self.cfg.enc_in:
            raise ValueError(f"Expected [B, {self.cfg.seq_len}, {self.cfg.enc_in}], got {tuple(x.shape)}")

        if self.revin is not None:
            x = self.revin.norm(x)

        # [B, L, C] -> [B, C, L] -> patches [B, C, P, PL]
        z = x.permute(0, 2, 1)
        patches = z.unfold(dimension=-1, size=self.cfg.patch_len, step=self.cfg.stride)
        B, C, P, PL = patches.shape
        tokens = patches.reshape(B * C, P, PL)  # channel-independent batch

        h = self.patch_embed(tokens)
        h = self.pos_enc(h)
        h = self.dropout(h)
        h = self.encoder(h)

        h_flat = h.reshape(B * C, -1)
        y_ch = self.channel_head(h_flat).reshape(B, C)  # [B, C]

        y = self.scalar_head(y_ch)  # [B, 1]
        return y.squeeze(-1)


__all__ = ["PatchTSTSOHModel"]

# ------------------------------------------------------------------------------
# Model 8: iTransformer (Liu et al., ICLR 2024 Spotlight)
# ------------------------------------------------------------------------------
"""iTransformer baseline adapted for NASA SOH seq-to-one regression.

Provenance (official upstream reference):
- Paper: "iTransformer: Inverted Transformers Are Effective for Time Series Forecasting" (ICLR 2024 Spotlight)
  - Paper PDF: https://proceedings.iclr.cc/paper_files/paper/2024/file/2ea18fdc667e0ef2ad82b2b4d65147ad-Paper-Conference.pdf
- Official repo: https://github.com/thuml/iTransformer
- Upstream commit (HEAD verified 2026-09-15): c2426e68ca13f74aaec08045c5c724d8ad328124
- License: MIT (per upstream repo)

Core architecture preserved? YES:
- **Inverted tokenization**: variates are tokens (N tokens), time points are token features.
- **Encoder-only Transformer**: native Transformer modules operate over variate tokens.

Task adaptation (minimal):
- Forecasting head replaced with a **seq-to-one regression head** for SOH.
- No timestamp covariates (`x_mark`) are used in this project; we follow upstream behavior for `x_mark=None`.
"""


from dataclasses import dataclass

import torch
from torch import nn


@dataclass(frozen=True)
class ITransformerConfig:
    seq_len: int = 512
    enc_in: int = 4          # number of variates/tokens
    d_model: int = 64
    n_heads: int = 2
    e_layers: int = 3
    d_ff: int = 128
    dropout: float = 0.0
    use_norm: bool = True    # upstream-style per-sample normalization
    pooling: str = "mean"    # token pooling over variates


class _DataEmbeddingInverted(nn.Module):
    """Upstream DataEmbedding_inverted (simplified): linear map Time->d_model per variate token."""

    def __init__(self, c_in: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.value_embedding = nn.Linear(c_in, d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, N] -> [B, N, L] -> [B, N, d_model]
        x = x.permute(0, 2, 1)
        x = self.value_embedding(x)
        return self.dropout(x)


class ITransformerSOHModel(nn.Module):
    """iTransformer-style inverted Transformer encoder for SOH regression: [B, 512, 4] -> [B]."""

    def __init__(
        self,
        seq_len: int = 512,
        enc_in: int = 4,
        d_model: int = 64,
        n_heads: int = 2,
        e_layers: int = 3,
        d_ff: int = 128,
        dropout: float = 0.0,
        use_norm: bool = True,
        pooling: str = "mean",
        head_hidden_dim: int = 64,
    ) -> None:
        super().__init__()
        self.cfg = ITransformerConfig(
            seq_len=seq_len,
            enc_in=enc_in,
            d_model=d_model,
            n_heads=n_heads,
            e_layers=e_layers,
            d_ff=d_ff,
            dropout=dropout,
            use_norm=use_norm,
            pooling=pooling,
        )

        # Embedding: invert and linearly embed per variate token
        self.enc_embedding = _DataEmbeddingInverted(c_in=seq_len, d_model=d_model, dropout=dropout)

        # Encoder-only Transformer over variate tokens (token length = enc_in = 4)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=e_layers)

        self.head = nn.Sequential(
            nn.Linear(d_model, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, N] where N=4
        if x.ndim != 3 or x.shape[1] != self.cfg.seq_len or x.shape[2] != self.cfg.enc_in:
            raise ValueError(f"Expected [B, {self.cfg.seq_len}, {self.cfg.enc_in}], got {tuple(x.shape)}")

        if self.cfg.use_norm:
            means = x.mean(dim=1, keepdim=True).detach()
            x0 = x - means
            stdev = torch.sqrt(torch.var(x0, dim=1, keepdim=True, unbiased=False) + 1e-5)
            x0 = x0 / stdev
        else:
            x0 = x

        # Embed and encode over variate tokens
        # embedding expects [B, L, N] but internally inverts to [B, N, L]
        enc_in = self.enc_embedding(x0)          # [B, N, d_model]
        enc_out = self.encoder(enc_in)           # [B, N, d_model]

        # Pool over tokens (variates)
        if self.cfg.pooling == "mean":
            pooled = enc_out.mean(dim=1)
        elif self.cfg.pooling == "cls":
            # optional: treat the first variate token as a representative token
            pooled = enc_out[:, 0, :]
        else:
            raise ValueError(f"Unknown pooling='{self.cfg.pooling}'.")

        y = self.head(pooled)  # [B, 1]
        return y.squeeze(-1)


__all__ = ["ITransformerSOHModel"]

# ------------------------------------------------------------------------------
# Model 9: QLSTM (Wang and Kebede hybrid QNN+LSTM baseline)
# ------------------------------------------------------------------------------
"""QLSTM baseline adapted from nasa-te-q-transformer-qlstm.ipynb.

Architecture (hybrid quantum-classical, NOT gate-level):
- Per-timestep linear projection: [B, L, 4] -> [B, L, n_qubits]
- PennyLane AngleEmbedding(Y) + BasicEntanglerLayers QNN (TorchLayer)
- Classical LSTM over quantum features
- Last/mean pooling + MLP head -> scalar SOH

Input/output contract matches all other E05 baselines: [B, 512, 4] -> [B].
"""


def _build_qnn_layer(n_qubits: int = 4, n_q_layers: int = 2):
    """Shared AngleEmbedding + BasicEntanglerLayers TorchLayer used by QLSTM and QNN-GRU."""
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def qnode(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    weight_shapes = {"weights": (n_q_layers, n_qubits)}
    return qml.qnn.TorchLayer(qnode, weight_shapes)


class QLSTMModel(nn.Module):
    """Hybrid QNN front-end + classical LSTM for SOH regression: [B, 512, 4] -> [B]."""

    def __init__(
        self,
        d_model: int = 64,
        hidden_size: int = 64,
        n_layers: int = 1,
        bidirectional: bool = False,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
        use_projection: bool = True,
        pooling: str = "last",
        n_qubits: int = 4,
        n_q_layers: int = 2,
    ) -> None:
        super().__init__()
        if pooling not in {"last", "mean"}:
            raise ValueError("pooling must be either 'last' or 'mean'.")
        self.d_model = d_model
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.bidirectional = bidirectional
        self.pooling = pooling
        self.use_projection = use_projection
        self.n_qubits = n_qubits
        self.n_q_layers = n_q_layers
        self.qnn_in_dim = n_qubits
        self.qnn_device = torch.device("cpu")
        self.input_projection = nn.Linear(4, self.qnn_in_dim) if use_projection else nn.Identity()
        self.qnn_layer = _build_qnn_layer(n_qubits=n_qubits, n_q_layers=n_q_layers)
        self.qnn_layer.to(self.qnn_device)
        self.qnn_out_projection = nn.Linear(n_qubits, d_model)
        lstm_dropout = dropout if n_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=lstm_dropout,
        )
        pool_dim = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.Linear(pool_dim, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3 or x.shape[-1] != 4:
            raise ValueError(f"Expected [B, L, 4], got {tuple(x.shape)}")
        batch_size, seq_len, _ = x.shape
        x = self.input_projection(x)
        q_dev = next(self.qnn_layer.parameters()).device
        x_q = x.reshape(batch_size * seq_len, -1).to(q_dev)
        qnn_features = self.qnn_layer(x_q)
        qnn_features = qnn_features.to(x.device)
        qnn_features = self.qnn_out_projection(qnn_features)
        qnn_features = qnn_features.view(batch_size, seq_len, -1)
        outputs, (h_n, _) = self.lstm(qnn_features)
        if self.pooling == "mean":
            pooled = outputs.mean(dim=1)
        else:
            if self.bidirectional:
                forward_last = h_n[-2]
                backward_last = h_n[-1]
                pooled = torch.cat([forward_last, backward_last], dim=1)
            else:
                pooled = h_n[-1]
        return self.head(pooled).squeeze(-1)


__all__ = ["QLSTMModel"]

# ------------------------------------------------------------------------------
# Model 10: QNN-GRU (Soon and Soon hybrid QNN+GRU baseline)
# ------------------------------------------------------------------------------
"""QNN-GRU baseline adapted from nasa-te-q-transformer-qnn-gru.ipynb.

Architecture (hybrid quantum-classical, NOT gate-level):
- Per-timestep linear projection: [B, L, 4] -> [B, L, n_qubits]
- Shared PennyLane AngleEmbedding(Y) + BasicEntanglerLayers QNN (TorchLayer)
- Classical GRU over quantum features
- Last/mean pooling + MLP head -> scalar SOH

Input/output contract matches all other E05 baselines: [B, 512, 4] -> [B].
"""


class QNNGRUModel(nn.Module):
    """Hybrid QNN front-end + classical GRU for SOH regression: [B, 512, 4] -> [B]."""

    def __init__(
        self,
        d_model: int = 64,
        hidden_size: int = 64,
        n_layers: int = 1,
        bidirectional: bool = False,
        dropout: float = 0.0,
        head_hidden_dim: int = 64,
        use_projection: bool = True,
        pooling: str = "last",
        n_qubits: int = 4,
        n_q_layers: int = 2,
    ) -> None:
        super().__init__()
        if pooling not in {"last", "mean"}:
            raise ValueError("pooling must be either 'last' or 'mean'.")
        self.d_model = d_model
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.bidirectional = bidirectional
        self.pooling = pooling
        self.use_projection = use_projection
        self.n_qubits = n_qubits
        self.n_q_layers = n_q_layers
        self.qnn_in_dim = n_qubits
        self.qnn_device = torch.device("cpu")
        self.input_projection = nn.Linear(4, self.qnn_in_dim) if use_projection else nn.Identity()
        self.qnn_layer = _build_qnn_layer(n_qubits=n_qubits, n_q_layers=n_q_layers)
        self.qnn_layer.to(self.qnn_device)
        self.qnn_out_projection = nn.Linear(n_qubits, d_model)
        gru_dropout = dropout if n_layers > 1 else 0.0
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=hidden_size,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=gru_dropout,
        )
        pool_dim = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.Linear(pool_dim, head_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3 or x.shape[-1] != 4:
            raise ValueError(f"Expected [B, L, 4], got {tuple(x.shape)}")
        batch_size, seq_len, _ = x.shape
        x = self.input_projection(x)
        q_dev = next(self.qnn_layer.parameters()).device
        x_q = x.reshape(batch_size * seq_len, -1).to(q_dev)
        qnn_features = self.qnn_layer(x_q)
        qnn_features = qnn_features.to(x.device)
        qnn_features = self.qnn_out_projection(qnn_features)
        qnn_features = qnn_features.view(batch_size, seq_len, -1)
        outputs, hidden = self.gru(qnn_features)
        if self.pooling == "mean":
            pooled = outputs.mean(dim=1)
        else:
            if self.bidirectional:
                forward_last = hidden[-2]
                backward_last = hidden[-1]
                pooled = torch.cat([forward_last, backward_last], dim=1)
            else:
                pooled = hidden[-1]
        return self.head(pooled).squeeze(-1)


__all__ = ["QLSTMModel", "QNNGRUModel"]

print("[Models] All 10 baseline architectures successfully compiled.")

"""TE-Q-Transformer architecture copied from the NASA ablation notebook.

Source: notebooks/nasa/nasa_teq_component_ablation_study.ipynb model cell.
This module is used only for frozen NASA → CALCE inference. It does not retrain
NASA weights and is not a substitute for the original notebooks.
"""


from dataclasses import dataclass

import pennylane as qml
import torch
from torch import nn


@dataclass(frozen=True)
class TEQTransformerConfig:
    input_dim: int = 4
    seq_len: int = 512
    quantum_dim: int = 4
    d_model: int = 64
    n_heads: int = 2
    n_layers: int = 3
    dim_feedforward: int = 64
    dropout: float = 0.0
    q_device: str = "default.qubit"
    entangler_layers: int = 1
    entangler_type: str = "basic"
    use_pauli_feature_map: bool = False
    feature_map_reps: int = 1
    feature_map_entangle: bool = True
    use_cls_token: bool = True
    pooling: str = "cls"
    use_positional_encoding: bool = True
    use_temporal_smooth: bool = True
    temporal_kernel_size: int = 3
    use_gru_smoother: bool = False
    gru_num_layers: int = 1
    gru_dropout: float = 0.0
    head_hidden_dim: int = 64
    use_residual_mlp: bool = False
    residual_mlp_dim: int = 128


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 1024) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-torch.log(torch.tensor(10000.0)) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class QuantumEmbeddingLayer(nn.Module):
    def __init__(
        self,
        n_qubits: int = 4,
        q_device: str = "default.qubit",
        entangler_layers: int = 1,
        entangler_type: str = "basic",
        use_pauli_feature_map: bool = False,
        feature_map_reps: int = 1,
        feature_map_entangle: bool = True,
    ) -> None:
        super().__init__()
        self.n_qubits = n_qubits
        self.R = 8.314462618
        self.T_ref = 298.15
        self.entangler_type = entangler_type
        self.use_pauli_feature_map = use_pauli_feature_map
        self.feature_map_reps = feature_map_reps
        self.feature_map_entangle = feature_map_entangle
        self.Ea_sei = nn.Parameter(torch.tensor(3.0, dtype=torch.float32))
        self.Ea_pl = nn.Parameter(torch.tensor(3.0, dtype=torch.float32))

        self.entangler_weights = nn.Parameter(
            0.01 * torch.randn(entangler_layers, n_qubits, dtype=torch.float32)
        )
        self.entangler_rzz = nn.Parameter(
            0.01 * torch.randn(entangler_layers, max(1, n_qubits - 1), dtype=torch.float32)
        )
        dev = qml.device(q_device, wires=n_qubits)

        def _apply_pauli_feature_map(inputs: torch.Tensor) -> None:
            for _ in range(self.feature_map_reps):
                for i in range(n_qubits):
                    qml.RX(inputs[:, i], wires=i)
                    qml.RY(inputs[:, i], wires=i)
                if self.feature_map_entangle:
                    for i in range(n_qubits - 1):
                        qml.CZ(wires=[i, i + 1])

        def _apply_basic_entangler(entangler_weights: torch.Tensor) -> None:
            qml.BasicEntanglerLayers(entangler_weights, wires=range(n_qubits))

        def _apply_rich_entangler(
            entangler_weights: torch.Tensor, entangler_rzz: torch.Tensor
        ) -> None:
            for layer in range(entangler_weights.shape[0]):
                for q in range(n_qubits):
                    qml.RZ(entangler_weights[layer, q], wires=q)
                for q in range(n_qubits - 1):
                    qml.CNOT(wires=[q, q + 1])
                for q in range(n_qubits - 1):
                    qml.IsingZZ(entangler_rzz[layer, q], wires=[q, q + 1])
                if n_qubits >= 3:
                    qml.Toffoli(wires=[0, 1, 2])
                    if n_qubits >= 4:
                        qml.Toffoli(wires=[1, 2, 3])
                for q in range(n_qubits):
                    qml.Hadamard(wires=q)
                    qml.RZ(entangler_weights[layer, q], wires=q)

        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def circuit(
            inputs: torch.Tensor,
            entangler_weights: torch.Tensor,
            entangler_rzz: torch.Tensor,
        ):
            if self.use_pauli_feature_map:
                _apply_pauli_feature_map(inputs)
            qml.RY(inputs[:, 0], wires=0)
            qml.RY(inputs[:, 1], wires=1)
            qml.RY(inputs[:, 2], wires=2)
            qml.RY(inputs[:, 3], wires=3)
            if self.entangler_type == "rich":
                _apply_rich_entangler(entangler_weights, entangler_rzz)
            else:
                _apply_basic_entangler(entangler_weights)
            return tuple(qml.expval(qml.PauliZ(i)) for i in range(n_qubits))

        self.circuit = circuit

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2 or x.shape[1] != 4:
            raise ValueError(f"Expected [B_flat, 4], got {tuple(x.shape)}")

        out_device = x.device
        out_dtype = x.dtype

        voltage_angle = x[:, 0] * torch.pi
        current_angle = x[:, 1] * torch.pi
        time_angle = x[:, 3] * torch.pi

        temp_c = x[:, 2]
        temp_k = torch.clamp(temp_c + 273.15, min=1.0)
        inv_t = 1.0 / temp_k
        inv_t_ref = 1.0 / self.T_ref
        Ea_sei_actual = self.Ea_sei * 10000.0
        Ea_pl_actual = self.Ea_pl * 10000.0

        sei_term = torch.exp((Ea_sei_actual / self.R) * (inv_t_ref - inv_t))
        plating_term = torch.exp((Ea_pl_actual / self.R) * (inv_t - inv_t_ref))
        phi = sei_term + plating_term
        theta_temp = torch.pi * phi / 4.0

        angles = torch.stack([voltage_angle, current_angle, time_angle, theta_temp], dim=1)

        angles_cpu = angles.to("cpu")
        entangler_cpu = self.entangler_weights.to("cpu")
        entangler_rzz_cpu = self.entangler_rzz.to("cpu")
        q_out = self.circuit(angles_cpu, entangler_cpu, entangler_rzz_cpu)
        q_tensor = torch.stack(q_out, dim=1).to(device=out_device, dtype=out_dtype)
        return q_tensor


class TEQTransformer(nn.Module):
    def __init__(self, cfg: TEQTransformerConfig | None = None) -> None:
        super().__init__()
        self.cfg = cfg or TEQTransformerConfig()
        if self.cfg.input_dim != 4:
            raise ValueError("This model expects exactly 4 input features.")
        if self.cfg.quantum_dim != 4:
            raise ValueError("quantum_dim must be 4 to match the 4 input channels.")
        if self.cfg.d_model % self.cfg.n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads.")
        if self.cfg.d_model <= self.cfg.quantum_dim:
            raise ValueError("d_model should be larger than quantum_dim after projection.")
        if self.cfg.pooling not in {"cls", "mean"}:
            raise ValueError("pooling must be either 'cls' or 'mean'.")
        if self.cfg.use_cls_token and self.cfg.pooling != "cls":
            raise ValueError("use_cls_token=True requires pooling='cls'.")
        if not self.cfg.use_cls_token and self.cfg.pooling != "mean":
            raise ValueError("use_cls_token=False requires pooling='mean'.")
        if self.cfg.use_temporal_smooth and self.cfg.temporal_kernel_size % 2 == 0:
            raise ValueError(
                "temporal_kernel_size should be odd so the sequence length is preserved."
            )
        if self.cfg.use_gru_smoother and self.cfg.use_temporal_smooth:
            raise ValueError("Enable only one temporal module: GRU smoother or Conv1D smoother.")
        if self.cfg.entangler_type not in {"basic", "rich"}:
            raise ValueError("entangler_type must be 'basic' or 'rich'.")
        if self.cfg.head_hidden_dim <= 0:
            raise ValueError("head_hidden_dim must be positive.")

        self.quantum_embed = QuantumEmbeddingLayer(
            n_qubits=self.cfg.quantum_dim,
            q_device=self.cfg.q_device,
            entangler_layers=self.cfg.entangler_layers,
            entangler_type=self.cfg.entangler_type,
            use_pauli_feature_map=self.cfg.use_pauli_feature_map,
            feature_map_reps=self.cfg.feature_map_reps,
            feature_map_entangle=self.cfg.feature_map_entangle,
        )
        self.quantum_proj = nn.Linear(self.cfg.quantum_dim, self.cfg.d_model)

        self.cls_token = nn.Parameter(torch.randn(1, 1, self.cfg.d_model)) if self.cfg.use_cls_token else None
        self.pos_encoder = (
            PositionalEncoding(self.cfg.d_model, max_len=self.cfg.seq_len + 2)
            if self.cfg.use_positional_encoding
            else None
        )

        if self.cfg.use_temporal_smooth:
            self.temporal_smooth = nn.Conv1d(
                in_channels=self.cfg.d_model,
                out_channels=self.cfg.d_model,
                kernel_size=self.cfg.temporal_kernel_size,
                padding=self.cfg.temporal_kernel_size // 2,
                bias=False,
            )
        else:
            self.temporal_smooth = nn.Identity()

        if self.cfg.use_gru_smoother:
            self.gru_smoother = nn.GRU(
                input_size=self.cfg.d_model,
                hidden_size=self.cfg.d_model,
                num_layers=self.cfg.gru_num_layers,
                dropout=self.cfg.gru_dropout if self.cfg.gru_num_layers > 1 else 0.0,
                batch_first=True,
            )
        else:
            self.gru_smoother = None

        self.residual_mlp = (
            nn.Sequential(
                nn.Linear(self.cfg.d_model, self.cfg.residual_mlp_dim),
                nn.GELU(),
                nn.Linear(self.cfg.residual_mlp_dim, self.cfg.d_model),
            )
            if self.cfg.use_residual_mlp
            else None
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.cfg.d_model,
            nhead=self.cfg.n_heads,
            dim_feedforward=self.cfg.dim_feedforward,
            dropout=self.cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=self.cfg.n_layers)
        self.head = nn.Sequential(
            nn.Linear(self.cfg.d_model, self.cfg.head_hidden_dim),
            nn.GELU(),
            nn.Dropout(self.cfg.dropout),
            nn.Linear(self.cfg.head_hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3 or x.shape[-1] != 4:
            raise ValueError(f"Expected [B, L, 4], got {tuple(x.shape)}")
        batch_size, seq_len, _ = x.shape
        if seq_len != self.cfg.seq_len:
            raise ValueError(f"Expected sequence length {self.cfg.seq_len}, got {seq_len}.")

        x_flat = x.reshape(batch_size * seq_len, 4)
        q_features = self.quantum_embed(x_flat)
        q_features = self.quantum_proj(q_features)
        q_sequence = q_features.reshape(batch_size, seq_len, self.cfg.d_model)

        if self.gru_smoother is not None:
            q_sequence, _ = self.gru_smoother(q_sequence)

        q_sequence = q_sequence.transpose(1, 2)
        q_sequence = self.temporal_smooth(q_sequence)
        q_sequence = q_sequence.transpose(1, 2)

        if self.residual_mlp is not None:
            q_sequence = q_sequence + self.residual_mlp(q_sequence)

        if self.cls_token is not None:
            cls_tokens = self.cls_token.expand(batch_size, -1, -1)
            q_sequence = torch.cat([cls_tokens, q_sequence], dim=1)

        if self.pos_encoder is not None:
            q_sequence = self.pos_encoder(q_sequence)

        transformed = self.transformer(q_sequence)
        if self.cfg.pooling == "cls":
            pooled = transformed[:, 0]
        else:
            pooled = transformed.mean(dim=1)
        soh = self.head(pooled)
        return soh.squeeze(-1)


def rich_entangler_config() -> TEQTransformerConfig:
    """Config of the retained NASA ablation baseline `01_rich_entangler`."""
    return TEQTransformerConfig(entangler_type="rich")

# ==============================================================================
# PART C: E07 FIVE-MODEL REGISTRY (E05/E06 TRAINING HYPERPARAMETERS)
# ==============================================================================
MODELS_REGISTRY: Dict[str, Dict[str, Any]] = {
    "TE-Q-Transformer": {
        "family": "Proposed-Quantum-Foundation",
        "builder": lambda: TEQTransformer(rich_entangler_config()),
        "expected_params": 92554,
        "batch_size": 8,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_notebook_cell": "TE-Q is defined in src/models/teq_transformer.py; referenced by E05/E06. Not a baselineComparison class.",
        "source_implementation": "src/models/teq_transformer.py (rich_entangler_config)",
        "e05_e06_reference": "E06 expected_params=92554, batch_size=8, AdamW lr=1e-3, wd=1e-2, 80 epochs, patience 20",
        "e07_changes": "None to architecture. TE-Q weight_decay=0.01 is retained from the established E01/E05/E06 implementation/campaign and is not tuned during E07. (E01 original config used 0.05; E05/E06 campaign registry used 0.01.)",
    },
    "QNN-GRU": {
        "family": "Quantum-Recurrent",
        "builder": lambda: QNNGRUModel(),
        "expected_params": 29533,
        "batch_size": 16,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_notebook_cell": "baselineComparison.ipynb cell 4, class QNNGRUModel (Model 10)",
        "source_implementation": "baselineComparison.ipynb / Experiment/Baseline/QNN_GRU.py",
        "e05_e06_reference": "E06 expected_params=29533, batch_size=16 (quantum-recurrent protocol)",
        "e07_changes": "None. batch_size=16 is the pre-existing E05/E06 deviation vs classical batch_size=8.",
    },
    "iTransformer": {
        "family": "Attention",
        "builder": lambda: ITransformerSOHModel(),
        "expected_params": 137473,
        "batch_size": 8,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_notebook_cell": "baselineComparison.ipynb cell 4, class ITransformerSOHModel (Model 8)",
        "source_implementation": "baselineComparison.ipynb / Experiment/Baseline/iTransformer.py",
        "e05_e06_reference": "E06 expected_params=137473, batch_size=8",
        "e07_changes": "None.",
    },
    "Transformer": {
        "family": "Attention",
        "builder": lambda: TransformerModel(),
        "expected_params": 80257,
        "batch_size": 8,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_notebook_cell": "baselineComparison.ipynb cell 4, class TransformerModel (Model 6)",
        "source_implementation": "baselineComparison.ipynb / Experiment/Baseline/Transformer.py",
        "e05_e06_reference": "E06 expected_params=80257, batch_size=8",
        "e07_changes": "None.",
    },
    "PatchTST": {
        "family": "Attention",
        "builder": lambda: PatchTSTSOHModel(),
        "expected_params": 109962,
        "batch_size": 8,
        "max_epochs": 80,
        "patience": 20,
        "lr": 1e-3,
        "weight_decay": 1e-2,
        "source_notebook_cell": "baselineComparison.ipynb cell 4, class PatchTSTSOHModel (Model 7)",
        "source_implementation": "baselineComparison.ipynb / Experiment/Baseline/PatchTST.py",
        "e05_e06_reference": "E06 expected_params=109962, batch_size=8",
        "e07_changes": "None.",
    },
}

assert list(MODELS_REGISTRY.keys()) == EXECUTION_ORDER

print("=" * 90)
print(f"{'Model':20s} | {'Source in baselineComparison.ipynb':46s} | Reused unchanged?")
print("=" * 90)
print(f"{'TE-Q-Transformer':20s} | {'not a cell-4 class; src/models/teq_transformer.py':46s} | YES")
print(f"{'QNN-GRU':20s} | {'cell 4 / class QNNGRUModel':46s} | YES")
print(f"{'iTransformer':20s} | {'cell 4 / class ITransformerSOHModel':46s} | YES")
print(f"{'Transformer':20s} | {'cell 4 / class TransformerModel':46s} | YES")
print(f"{'PatchTST':20s} | {'cell 4 / class PatchTSTSOHModel':46s} | YES")
print("=" * 90)
print(f"[Model Registry] {len(MODELS_REGISTRY)} designated E07 models registered. Extra cell-4 classes are loaded but not trained.")


## 8. Seed Configuration


In [ ]:
# ==============================================================================
# SECTION 7: SEED MANAGEMENT
# ==============================================================================
def set_seed(seed: int) -> None:
    """Deterministic seed for Python, NumPy, PyTorch, and CUDA."""
    seed = int(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        pass


def run_dir(model_name: str, seed: int) -> Path:
    return CHECKPOINTS_DIR / model_name / f"seed_{seed}"


def safe_torch_load(path: Path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def release_cuda(*objs) -> None:
    for obj in objs:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print(f"[Seed management] Fixed SEEDS = {SEEDS}")
print("[Seed management] set_seed() controls random / numpy / torch / CUDA; cudnn.deterministic=True, benchmark=False.")


## 9. Pre-flight Verification

If any designated model fails, FULL training must not start.


In [ ]:
# ==============================================================================
# SECTION 8: PRE-FLIGHT TESTS (ALL FIVE DESIGNATED MODELS)
# ==============================================================================
print("=" * 75)
print("E07 PRE-FLIGHT")
print("=" * 75)
all_preflight_passed = True
preflight_status: Dict[str, str] = {}

for model_name in EXECUTION_ORDER:
    info = MODELS_REGISTRY[model_name]
    try:
        set_seed(42)
        m = info["builder"]().to(DEVICE)
        tot_params = sum(p.numel() for p in m.parameters())
        trainable_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
        assert tot_params == info["expected_params"], (
            f"Param mismatch for {model_name}: expected {info['expected_params']}, got {tot_params}"
        )
        bx = torch.randn(2, 512, 4, device=DEVICE)
        bx[:, :, 2] = 24.0
        by = torch.tensor([0.95, 0.85], dtype=torch.float32, device=DEVICE)
        m.train()
        out = m(bx)
        assert out.shape == (2,), f"Output shape mismatch: {out.shape}"
        assert torch.all(torch.isfinite(out)), "Non-finite output"
        crit = nn.MSELoss()
        loss = crit(out, by)
        assert torch.isfinite(loss), "Non-finite loss"
        opt = optim.AdamW(m.parameters(), lr=info["lr"], weight_decay=info["weight_decay"])
        opt.zero_grad()
        loss.backward()
        for pname, p in m.named_parameters():
            if p.requires_grad and p.grad is not None:
                assert torch.all(torch.isfinite(p.grad)), f"Non-finite grad {pname}"
        opt.step()
        test_ckpt = CHECKPOINTS_DIR / f"_preflight_{model_name.replace('-', '_')}.pth"
        torch.save({"model_state": m.state_dict(), "optimizer_state": opt.state_dict(), "seed": 42}, test_ckpt)
        blob = safe_torch_load(test_ckpt, map_location=DEVICE)
        m.load_state_dict(blob["model_state"])
        if test_ckpt.exists():
            test_ckpt.unlink()
        m.eval()
        with torch.no_grad():
            eval_out = m(bx)
        assert eval_out.shape == (2,)
        metric_dict = compute_metrics(by.detach().cpu().numpy(), eval_out.detach().cpu().numpy())
        for k in ["RMSE", "MAE", "MAPE (%)", "R2", "MaxE"]:
            assert k in metric_dict
            if k != "R2":
                assert np.isfinite(metric_dict[k]), f"Invalid metric {k}"
        print(f"MODEL: {model_name:18s} | Params: {tot_params:7,d} | Status: PASS")
        preflight_status[model_name] = "PASS"
        release_cuda(m, opt, out, eval_out, loss, bx, by)
    except Exception as e:
        print(f"MODEL: {model_name:18s} | Status: FAIL | Error: {e}")
        preflight_status[model_name] = f"FAIL: {e}"
        all_preflight_passed = False

print("=" * 75)
for mname in EXECUTION_ORDER:
    print(f"  {mname:18s} -> {preflight_status.get(mname, 'MISSING')}")
if all_preflight_passed:
    print("ALL 5 MODELS PRE-FLIGHT TEST: PASS")
    print("ALL PRE-FLIGHT CHECKS PASSED")
else:
    print("PRE-FLIGHT TEST: FAIL")
    raise RuntimeError("One or more models failed pre-flight. Do not start FULL.")
print("=" * 75)


## 10. Training and Evaluation Engine

Checkpoint selection uses **training MSE only**. Test RMSE is never used to pick the best epoch.


In [ ]:
# ==============================================================================
# SECTION 9: REUSABLE TRAINING FUNCTION
# ==============================================================================
def train_one_run(
    model_name: str,
    model_info: Dict[str, Any],
    train_loader: DataLoader,
    seed: int,
    run_mode: str,
) -> Tuple[Dict[str, torch.Tensor], List[Dict[str, Any]], int, float, float]:
    set_seed(seed)
    ckpt_folder = run_dir(model_name, seed)
    ckpt_folder.mkdir(parents=True, exist_ok=True)
    best_path = ckpt_folder / "model_best.pth"
    latest_path = ckpt_folder / "model_latest.pth"

    model = model_info["builder"]().to(DEVICE)
    max_epochs = 1 if run_mode == "DRY_RUN" else int(model_info["max_epochs"])
    patience = int(model_info["patience"])
    optimizer = optim.AdamW(model.parameters(), lr=model_info["lr"], weight_decay=model_info["weight_decay"])
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10, min_lr=1e-6)
    criterion = nn.MSELoss()

    best_loss = float("inf")
    best_epoch = 0
    patience_counter = 0
    history_records: List[Dict[str, Any]] = []
    start_time = time.time()

    cfg_blob = {
        "model_name": model_name,
        "family": model_info["family"],
        "batch_size": model_info["batch_size"],
        "max_epochs": model_info["max_epochs"],
        "patience": model_info["patience"],
        "lr": model_info["lr"],
        "weight_decay": model_info["weight_decay"],
        "expected_params": model_info["expected_params"],
    }

    for epoch in range(1, max_epochs + 1):
        epoch_t0 = time.time()
        model.train()
        running_loss = 0.0
        n_batches = 0
        for bx, by, _cyc in train_loader:
            bx = bx.to(DEVICE)
            by = by.to(DEVICE)
            optimizer.zero_grad()
            preds = model(bx)
            loss = criterion(preds, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += float(loss.item())
            n_batches += 1
            if run_mode == "DRY_RUN":
                break
        epoch_loss = running_loss / max(n_batches, 1)
        epoch_time = time.time() - epoch_t0
        current_lr = float(optimizer.param_groups[0]["lr"])
        history_records.append({
            "model": model_name,
            "seed": int(seed),
            "epoch": epoch,
            "train_loss": epoch_loss,
            "validation_loss": float("nan"),
            "learning_rate": current_lr,
            "elapsed_seconds": epoch_time,
        })
        if run_mode == "FULL":
            scheduler.step(epoch_loss)

        payload = {
            "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "epoch": epoch,
            "best_train_loss": min(best_loss, epoch_loss),
            "seed": int(seed),
            "model_name": model_name,
            "model_configuration": cfg_blob,
        }
        torch.save(payload, latest_path)

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_epoch = epoch
            patience_counter = 0
            torch.save(payload, best_path)
        else:
            patience_counter += 1

        if epoch % 10 == 0 or epoch == 1 or epoch == max_epochs:
            print(
                f"  [{model_name} seed={seed}] Epoch {epoch:2d}/{max_epochs} | "
                f"train MSE {epoch_loss:.6f} | lr {current_lr:.1e} | "
                f"best {best_loss:.6f} (ep {best_epoch}) | {epoch_time:.2f}s"
            )
        if run_mode == "FULL" and patience_counter >= patience:
            print(f"  [{model_name} seed={seed}] Early stopping at epoch {epoch} (training-loss patience={patience}).")
            break

    total_training_sec = time.time() - start_time
    blob = safe_torch_load(best_path, map_location="cpu")
    best_state = blob["model_state"]
    release_cuda(model, optimizer, scheduler)
    return best_state, history_records, best_epoch, best_loss, total_training_sec


### 10b. Evaluation function


In [ ]:
# ==============================================================================
# SECTION 10: REUSABLE EVALUATION FUNCTION
# ==============================================================================
EVAL_TYPE = {
    "B0018": "unseen_cell",
    "B0032": "unseen_cell",
    "B0053_test": "temporal_extrapolation",
}


def evaluate_one_run(
    model_name: str,
    model_info: Dict[str, Any],
    best_state: Dict[str, torch.Tensor],
    test_loaders: Dict[str, DataLoader],
    seed: int,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], float]:
    set_seed(seed)
    model = model_info["builder"]().to(DEVICE)
    model.load_state_dict(best_state)
    model.eval()
    cell_rows: List[Dict[str, Any]] = []
    pred_rows: List[Dict[str, Any]] = []
    t0 = time.time()
    with torch.no_grad():
        for cell_id, loader in test_loaders.items():
            eval_type = EVAL_TYPE[cell_id]
            y_t, y_p, cycles, sidx = [], [], [], []
            local_i = 0
            for bx, by, cyc in loader:
                bx = bx.to(DEVICE)
                pred = model(bx)
                yt = by.cpu().numpy().reshape(-1)
                yp = pred.cpu().numpy().reshape(-1)
                cc = np.asarray(cyc).reshape(-1)
                y_t.append(yt)
                y_p.append(yp)
                cycles.append(cc)
                for _ in range(len(yt)):
                    sidx.append(local_i)
                    local_i += 1
            yt_arr = np.concatenate(y_t)
            yp_arr = np.concatenate(y_p)
            cyc_arr = np.concatenate(cycles)
            m = compute_metrics(yt_arr, yp_arr)
            cell_rows.append({
                "model": model_name,
                "family": model_info["family"],
                "seed": int(seed),
                "cell": cell_id,
                "evaluation_type": eval_type,
                "n_samples": int(len(yt_arr)),
                "rmse": m["RMSE"],
                "mae": m["MAE"],
                "mape": m["MAPE (%)"],
                "r2": m["R2"],
                "max_error": m["MaxE"],
            })
            for t_val, p_val, cid, si in zip(yt_arr, yp_arr, cyc_arr, sidx):
                err = float(p_val) - float(t_val)
                pred_rows.append({
                    "model": model_name,
                    "family": model_info["family"],
                    "seed": int(seed),
                    "cell": cell_id,
                    "evaluation_type": eval_type,
                    "cycle_id": int(cid),
                    "sample_index": int(si),
                    "true_soh": float(t_val),
                    "pred_soh": float(p_val),
                    "error": err,
                    "abs_error": abs(err),
                    "squared_error": err ** 2,
                })
            print(
                f"    {cell_id:12s} ({eval_type:24s}) n={len(yt_arr):3d} | "
                f"RMSE {m['RMSE']:.6f} | MAE {m['MAE']:.6f} | R2 {m['R2']:.6f}"
            )
    infer_time = time.time() - t0
    release_cuda(model)
    return cell_rows, pred_rows, infer_time


## 11. 25-Run Sequential Execution

Order: TE-Q-Transformer, QNN-GRU, iTransformer, Transformer, PatchTST. For each model: seeds 42, 43, 44, 45, 46. One run at a time. Completed runs are skipped only after checkpoint + predictions + metrics all validate.


In [ ]:
# ==============================================================================
# SECTION 11: 25-RUN SEQUENTIAL EXECUTION (RESUMABLE)
# ==============================================================================
def _read_csv_if_exists(path: Path) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path)
    return pd.DataFrame()


def run_is_complete(model_name: str, seed: int) -> bool:
    best_path = run_dir(model_name, seed) / "model_best.pth"
    if not best_path.exists():
        return False
    try:
        blob = safe_torch_load(best_path, map_location="cpu")
        if "model_state" not in blob:
            return False
    except Exception:
        return False
    pred_df = _read_csv_if_exists(PREDICTIONS_DIR / "E07_predictions.csv")
    run_df = _read_csv_if_exists(METRICS_DIR / "E07_run_metrics.csv")
    cell_df = _read_csv_if_exists(METRICS_DIR / "E07_cell_metrics.csv")
    if pred_df.empty or run_df.empty or cell_df.empty:
        return False
    if "seed" not in pred_df.columns:
        return False
    n_pred = int(((pred_df["model"] == model_name) & (pred_df["seed"] == seed)).sum())
    n_run = int(((run_df["model"] == model_name) & (run_df["seed"] == seed)).sum())
    n_cell = int(((cell_df["model"] == model_name) & (cell_df["seed"] == seed)).sum())
    cfg_df = _read_csv_if_exists(CONFIGS_DIR / "E07_model_configs.csv")
    n_cfg = 0
    if not cfg_df.empty and "seed" in cfg_df.columns:
        n_cfg = int(((cfg_df["model"] == model_name) & (cfg_df["seed"] == seed)).sum())
    latest_path = run_dir(model_name, seed) / "model_latest.pth"
    return (n_pred == 187) and (n_run == 1) and (n_cell == 3) and (n_cfg == 1) and latest_path.exists()


def save_live_tables(run_rows, cell_rows, pred_rows, hist_rows, cfg_rows) -> None:
    pd.DataFrame(run_rows).to_csv(METRICS_DIR / "E07_run_metrics.csv", index=False)
    pd.DataFrame(cell_rows).to_csv(METRICS_DIR / "E07_cell_metrics.csv", index=False)
    pd.DataFrame(pred_rows).to_csv(PREDICTIONS_DIR / "E07_predictions.csv", index=False)
    pd.DataFrame(hist_rows).to_csv(TRAINING_DIR / "E07_training_history.csv", index=False)
    pd.DataFrame(cfg_rows).to_csv(CONFIGS_DIR / "E07_model_configs.csv", index=False)


try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL).decode().strip()
except Exception:
    git_commit = "N/A"

all_run_metrics: List[Dict[str, Any]] = []
all_cell_metrics: List[Dict[str, Any]] = []
all_predictions: List[Dict[str, Any]] = []
all_training_histories: List[Dict[str, Any]] = []
all_model_configs: List[Dict[str, Any]] = []
run_status: Dict[Tuple[str, int], str] = {}
provenance_runs: List[Dict[str, Any]] = []

# Load any already-complete artifacts for resume
_pred_ex = _read_csv_if_exists(PREDICTIONS_DIR / "E07_predictions.csv")
_run_ex = _read_csv_if_exists(METRICS_DIR / "E07_run_metrics.csv")
_cell_ex = _read_csv_if_exists(METRICS_DIR / "E07_cell_metrics.csv")
_hist_ex = _read_csv_if_exists(TRAINING_DIR / "E07_training_history.csv")
_cfg_ex = _read_csv_if_exists(CONFIGS_DIR / "E07_model_configs.csv")

planned = [(m, s) for m in EXECUTION_ORDER for s in SEEDS]
assert len(planned) == 25

print("=" * 75)
print(f"E07 {RUN_MODE} : {len(planned)} planned runs")
print("=" * 75)

wall0 = time.time()
run_index = 0
for model_name, seed in planned:
    run_index += 1
    info = MODELS_REGISTRY[model_name]
    print("\n" + "=" * 64)
    print(f"RUN {run_index:02d} / 25")
    print(f"Model: {model_name}")
    print(f"Seed: {seed}")
    print(f"Device: {DEVICE}")
    print("=" * 64)

    if run_is_complete(model_name, seed):
        print(f"SKIPPING VERIFIED COMPLETED RUN:\n{model_name} / seed {seed}")
        run_status[(model_name, seed)] = "SKIPPED_COMPLETE"
        sub_run = _run_ex[(_run_ex["model"] == model_name) & (_run_ex["seed"] == seed)]
        sub_cell = _cell_ex[(_cell_ex["model"] == model_name) & (_cell_ex["seed"] == seed)]
        sub_pred = _pred_ex[(_pred_ex["model"] == model_name) & (_pred_ex["seed"] == seed)]
        if not _hist_ex.empty and "seed" in _hist_ex.columns:
            sub_hist = _hist_ex[(_hist_ex["model"] == model_name) & (_hist_ex["seed"] == seed)]
            all_training_histories.extend(sub_hist.to_dict("records"))
        if not _cfg_ex.empty and "seed" in _cfg_ex.columns:
            sub_cfg = _cfg_ex[(_cfg_ex["model"] == model_name) & (_cfg_ex["seed"] == seed)]
            all_model_configs.extend(sub_cfg.to_dict("records"))
        all_run_metrics.extend(sub_run.to_dict("records"))
        all_cell_metrics.extend(sub_cell.to_dict("records"))
        all_predictions.extend(sub_pred.to_dict("records"))
        continue

    set_seed(seed)
    train_loader, test_loaders, scaler, split_idx = get_nasa_dataloaders(
        DATA_ROOT, batch_size=info["batch_size"], seed=seed, num_workers=0
    )
    assert "B0018" not in NASA_FULL_TRAIN_CELLS
    assert "B0032" not in NASA_FULL_TRAIN_CELLS
    assert split_idx == 37
    assert int(scaler.n_samples_seen_) == N_TRAIN_EXPECTED * SEQUENCE_LENGTH

    best_state, history, best_epoch, best_loss, train_time = train_one_run(
        model_name, info, train_loader, seed, RUN_MODE
    )
    all_training_histories.extend(history)

    cell_rows, pred_rows, infer_time = evaluate_one_run(
        model_name, info, best_state, test_loaders, seed
    )
    all_cell_metrics.extend(cell_rows)
    all_predictions.extend(pred_rows)

    macro_rmse = float(np.mean([c["rmse"] for c in cell_rows]))
    macro_mae = float(np.mean([c["mae"] for c in cell_rows]))
    macro_mape = float(np.mean([c["mape"] for c in cell_rows]))
    macro_r2 = float(np.mean([c["r2"] for c in cell_rows]))
    macro_max_e = float(np.max([c["max_error"] for c in cell_rows]))

    tmp_model = info["builder"]()
    tot_params = int(sum(p.numel() for p in tmp_model.parameters()))
    trainable_params = int(sum(p.numel() for p in tmp_model.parameters() if p.requires_grad))
    release_cuda(tmp_model)

    all_run_metrics.append({
        "model": model_name,
        "family": info["family"],
        "seed": int(seed),
        "overall_rmse": macro_rmse,
        "overall_mae": macro_mae,
        "overall_mape": macro_mape,
        "overall_r2": macro_r2,
        "overall_max_error": macro_max_e,
        "trainable_params": trainable_params,
        "total_params": tot_params,
        "training_time_seconds": float(train_time),
        "inference_time_seconds": float(infer_time),
        "best_epoch": int(best_epoch),
        "device": str(DEVICE),
    })
    all_model_configs.append({
        "model": model_name,
        "family": info["family"],
        "seed": int(seed),
        "parameter_count": tot_params,
        "sequence_length": SEQUENCE_LENGTH,
        "features": "V,I,T_C,Time_norm",
        "batch_size": info["batch_size"],
        "learning_rate": info["lr"],
        "weight_decay": info["weight_decay"],
        "max_epochs": info["max_epochs"],
        "scheduler": "ReduceLROnPlateau(factor=0.5,patience=10)",
        "stopping_rule": "early stop on training MSE, patience=20; best checkpoint = min training MSE (never test RMSE)",
        "configuration_notes": info["e07_changes"],
    })
    provenance_runs.append({
        "model": model_name,
        "seed": int(seed),
        "parameter_count": tot_params,
        "hyperparameters": {
            "batch_size": info["batch_size"],
            "learning_rate": info["lr"],
            "weight_decay": info["weight_decay"],
            "max_epochs": info["max_epochs"],
            "patience": info["patience"],
            "optimizer": "AdamW",
            "scheduler": "ReduceLROnPlateau(factor=0.5, patience=10)",
            "loss": "MSE",
            "grad_clip_norm": 1.0,
        },
        "source_implementation": info["source_implementation"],
        "source_notebook_cell": info["source_notebook_cell"],
        "best_epoch": int(best_epoch),
        "best_train_loss": float(best_loss),
        "training_time_seconds": float(train_time),
        "device": str(DEVICE),
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    })
    run_status[(model_name, seed)] = "COMPLETED"
    save_live_tables(all_run_metrics, all_cell_metrics, all_predictions, all_training_histories, all_model_configs)
    print(f"  [saved] live CSVs after {model_name} seed={seed}")

    del train_loader, test_loaders, scaler, best_state
    release_cuda()
    print("  [memory] model/optimizer/scheduler/loaders released; gc.collect(); cuda.empty_cache()")

wall_total = time.time() - wall0
print(f"\n[E07] Sequential loop finished in {wall_total:.1f}s | completed+skipped = {len(run_status)} / 25")

provenance_meta = {
    "experiment": "E07_MultiSeed_Robustness",
    "research_question": "How sensitive are the observed E06 performance results to random initialization across a predefined set of five random seeds?",
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "models": EXECUTION_ORDER,
    "seeds": SEEDS,
    "n_planned_runs": 25,
    "dataset": "NASA Ames Li-ion Battery Aging Dataset",
    "training_cells": list(NASA_FULL_TRAIN_CELLS) + ["B0053_first_70% (cycles 0-36)"],
    "unseen_test_cells": list(NASA_FULL_TEST_CELLS),
    "temporal_extrapolation_cell": "B0053_final_30% (cycles 37-52)",
    "B0053_split": "cycles 0-36 train; cycles 37-52 test; total 53 cycles",
    "source_notebook": "baselineComparison.ipynb cell 4 + src/models/teq_transformer.py",
    "sequence_length": SEQUENCE_LENGTH,
    "features": FEATURES,
    "scaler_policy": "MinMaxScaler fitted STRICTLY on 660 training cycles; Temperature in unscaled Celsius",
    "pytorch_version": torch.__version__,
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "device": str(DEVICE),
    "run_mode": RUN_MODE,
    "git_commit": git_commit,
    "wall_time_seconds": wall_total,
    "runs": provenance_runs,
}
(PROVENANCE_DIR / "E07_provenance.json").write_text(json.dumps(provenance_meta, indent=2), encoding="utf-8")


## 12. Result Aggregation


In [ ]:
# ==============================================================================
# SECTION 12: RESULT AGGREGATION
# ==============================================================================
run_df = pd.DataFrame(all_run_metrics)
cell_df = pd.DataFrame(all_cell_metrics)
pred_df = pd.DataFrame(all_predictions)
hist_df = pd.DataFrame(all_training_histories)
cfg_df = pd.DataFrame(all_model_configs)

save_live_tables(all_run_metrics, all_cell_metrics, all_predictions, all_training_histories, all_model_configs)

print("E07_run_metrics.csv rows :", len(run_df), "(expected 25)")
print("E07_cell_metrics.csv rows:", len(cell_df), "(expected 75)")
print("E07_predictions.csv rows :", len(pred_df), "(expected 4675 = 25*187)")
print("Unique models:", sorted(run_df["model"].unique().tolist()) if len(run_df) else [])
print("Unique seeds :", sorted(run_df["seed"].unique().tolist()) if len(run_df) else [])
print(run_df[["model", "seed", "overall_rmse", "overall_mae", "overall_r2", "best_epoch"]].to_string(index=False) if len(run_df) else "no runs")

if RUN_MODE == "FULL":
    assert len(run_df) == 25, f"E07_run_metrics.csv must have 25 rows, got {len(run_df)}"
    assert len(cell_df) == 75, f"E07_cell_metrics.csv must have 75 rows, got {len(cell_df)}"
    assert len(pred_df) == 25 * 187, f"E07_predictions.csv must have 4675 rows, got {len(pred_df)}"
    assert sorted(run_df["model"].unique().tolist()) == sorted(EXECUTION_ORDER)
    assert sorted(int(s) for s in run_df["seed"].unique().tolist()) == SEEDS
    print("[Row-count assertions] PASS: 25 runs, 75 cell rows, 4675 predictions.")


## 13. Reconciliation

Metrics are recomputed from `E07_predictions.csv`. Mismatches are documented; stored files are not silently rewritten.


In [ ]:
# ==============================================================================
# SECTION 13: METRIC RECONCILIATION
# ==============================================================================
print("=" * 75)
print("E07 AUTOMATIC RECONCILIATION")
print("=" * 75)
preds_df = pd.read_csv(PREDICTIONS_DIR / "E07_predictions.csv")
cell_stored = pd.read_csv(METRICS_DIR / "E07_cell_metrics.csv")
run_stored = pd.read_csv(METRICS_DIR / "E07_run_metrics.csv")

recon_lines = [
    "================================================================================",
    "E07 AUTOMATIC RESULT RECONCILIATION REPORT",
    "================================================================================",
    f"Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}",
    f"RUN_MODE: {RUN_MODE}",
    f"Total prediction rows: {len(preds_df):,}",
    "Tolerance: 1e-5",
    "Mismatches are reported. Stored CSVs are NOT silently modified.",
    "================================================================================",
    "",
]
discrepancy_count = 0
for (m_name, seed, c_id), group in preds_df.groupby(["model", "seed", "cell"]):
    recomp = compute_metrics(group["true_soh"].values, group["pred_soh"].values)
    hit = cell_stored[
        (cell_stored["model"] == m_name) & (cell_stored["seed"] == seed) & (cell_stored["cell"] == c_id)
    ]
    if hit.empty:
        discrepancy_count += 1
        msg = f"MISMATCH: {m_name} seed={seed} cell={c_id} | missing stored cell row"
        print(msg)
        recon_lines.append(msg)
        continue
    matched = hit.iloc[0]
    rmse_diff = abs(recomp["RMSE"] - float(matched["rmse"]))
    mae_diff = abs(recomp["MAE"] - float(matched["mae"]))
    r2_a, r2_b = recomp["R2"], float(matched["r2"])
    if np.isnan(r2_a) and np.isnan(r2_b):
        r2_diff = 0.0
    else:
        r2_diff = abs(r2_a - r2_b)
    if rmse_diff > 1e-5 or mae_diff > 1e-5 or r2_diff > 1e-5:
        discrepancy_count += 1
        msg = (
            f"MISMATCH: {m_name:18s} seed={int(seed)} cell={c_id:12s} | "
            f"RMSE diff={rmse_diff:.2e} MAE diff={mae_diff:.2e} R2 diff={r2_diff:.2e}"
        )
        print(msg)
        recon_lines.append(msg)
    else:
        recon_lines.append(
            f"MATCH: {m_name:18s} seed={int(seed)} cell={c_id:12s} RMSE={float(matched['rmse']):.6f}"
        )

# overall vs mean of three cells
for (m_name, seed), g in cell_stored.groupby(["model", "seed"]):
    if len(g) != 3:
        continue
    rec_rmse = float(g["rmse"].mean())
    rec_mae = float(g["mae"].mean())
    rec_r2 = float(g["r2"].mean())
    hit = run_stored[(run_stored["model"] == m_name) & (run_stored["seed"] == seed)]
    if hit.empty:
        continue
    row = hit.iloc[0]
    d_rmse = abs(rec_rmse - float(row["overall_rmse"]))
    d_mae = abs(rec_mae - float(row["overall_mae"]))
    d_r2 = abs(rec_r2 - float(row["overall_r2"]))
    if d_rmse > 1e-5 or d_mae > 1e-5 or d_r2 > 1e-5:
        discrepancy_count += 1
        msg = f"MISMATCH overall: {m_name} seed={int(seed)} RMSE/MAE/R2 vs cell-mean"
        print(msg)
        recon_lines.append(msg)
    else:
        recon_lines.append(f"MATCH overall: {m_name:18s} seed={int(seed)}")

if discrepancy_count == 0:
    verdict = "PASS - ALL PREDICTIONS AND METRICS RECONCILED (diff < 1e-5)."
else:
    verdict = f"WARNING - FOUND {discrepancy_count} DISCREPANCIES. Stored files were not modified."
print("\n" + verdict)
recon_lines.append("")
recon_lines.append("VERDICT: " + verdict)
(REPORTS_DIR / "E07_Reconciliation_Report.txt").write_text("\n".join(recon_lines), encoding="utf-8")
RECON_VERDICT = verdict


### 13b. Multi-seed summary tables

These tables compute mean and sample SD across the five predefined seeds. They do not claim statistical significance.


In [ ]:
# ==============================================================================
# SECTION 14: MULTI-SEED SUMMARY (DESCRIPTIVE ONLY)
# ==============================================================================
def sample_std(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    if x.size < 2:
        return float("nan")
    return float(np.std(x, ddof=1))


def cv(mean_v: float, std_v: float) -> float:
    if mean_v is None or not np.isfinite(mean_v) or abs(mean_v) < 1e-12:
        return float("nan")
    return float(std_v / mean_v)


ms_rows = []
for model_name in EXECUTION_ORDER:
    sub = run_stored[run_stored["model"] == model_name]
    if sub.empty:
        continue
    means = {
        "rmse": float(sub["overall_rmse"].mean()),
        "mae": float(sub["overall_mae"].mean()),
        "mape": float(sub["overall_mape"].mean()),
        "r2": float(sub["overall_r2"].mean()),
        "max_error": float(sub["overall_max_error"].mean()),
    }
    stds = {
        "rmse": sample_std(sub["overall_rmse"].values),
        "mae": sample_std(sub["overall_mae"].values),
        "mape": sample_std(sub["overall_mape"].values),
        "r2": sample_std(sub["overall_r2"].values),
        "max_error": sample_std(sub["overall_max_error"].values),
    }
    ms_rows.append({
        "model": model_name,
        "n_seeds": int(sub["seed"].nunique()),
        "mean_rmse": means["rmse"],
        "std_rmse": stds["rmse"],
        "cv_rmse": cv(means["rmse"], stds["rmse"]),
        "mean_mae": means["mae"],
        "std_mae": stds["mae"],
        "cv_mae": cv(means["mae"], stds["mae"]),
        "mean_mape": means["mape"],
        "std_mape": stds["mape"],
        "cv_mape": cv(means["mape"], stds["mape"]),
        "mean_r2": means["r2"],
        "std_r2": stds["r2"],
        "mean_max_error": means["max_error"],
        "std_max_error": stds["max_error"],
    })
ms_df = pd.DataFrame(ms_rows)
ms_df.to_csv(METRICS_DIR / "E07_MultiSeed_Summary.csv", index=False)

cell_ms_rows = []
for model_name in EXECUTION_ORDER:
    for cell_id in ["B0018", "B0032", "B0053_test"]:
        sub = cell_stored[(cell_stored["model"] == model_name) & (cell_stored["cell"] == cell_id)]
        if sub.empty:
            continue
        cell_ms_rows.append({
            "model": model_name,
            "cell": cell_id,
            "evaluation_type": EVAL_TYPE[cell_id],
            "n_seeds": int(sub["seed"].nunique()),
            "mean_rmse": float(sub["rmse"].mean()),
            "std_rmse": sample_std(sub["rmse"].values),
            "mean_mae": float(sub["mae"].mean()),
            "std_mae": sample_std(sub["mae"].values),
            "mean_mape": float(sub["mape"].mean()),
            "std_mape": sample_std(sub["mape"].values),
            "mean_r2": float(sub["r2"].mean()),
            "std_r2": sample_std(sub["r2"].values),
        })
cell_ms_df = pd.DataFrame(cell_ms_rows)
cell_ms_df.to_csv(METRICS_DIR / "E07_MultiSeed_Cell_Summary.csv", index=False)

seed_rows = []
for model_name in EXECUTION_ORDER:
    for seed in SEEDS:
        sub = cell_stored[(cell_stored["model"] == model_name) & (cell_stored["seed"] == seed)]
        if sub.empty:
            continue
        rec = {"model": model_name, "seed": int(seed)}
        for cell_id, prefix in [("B0018", "B0018"), ("B0032", "B0032"), ("B0053_test", "B0053_test")]:
            row = sub[sub["cell"] == cell_id]
            if row.empty:
                rec[f"{prefix}_rmse"] = float("nan")
                rec[f"{prefix}_mae"] = float("nan")
                rec[f"{prefix}_r2"] = float("nan")
            else:
                rec[f"{prefix}_rmse"] = float(row.iloc[0]["rmse"])
                rec[f"{prefix}_mae"] = float(row.iloc[0]["mae"])
                rec[f"{prefix}_r2"] = float(row.iloc[0]["r2"])
        seed_rows.append(rec)
seed_df = pd.DataFrame(seed_rows)
seed_df.to_csv(METRICS_DIR / "E07_Seed_Wise_Detail.csv", index=False)

print("E07_MultiSeed_Summary.csv")
print(ms_df.to_string(index=False) if len(ms_df) else "(empty)")
print("\nE07_MultiSeed_Cell_Summary.csv rows:", len(cell_ms_df))
print("E07_Seed_Wise_Detail.csv rows:", len(seed_df))
print("\nInterpretation is deferred. These statistics are descriptive across the fixed five-seed set.")

if RUN_MODE == "FULL":
    assert len(ms_df) == 5, f"E07_MultiSeed_Summary.csv must have 5 rows, got {len(ms_df)}"
    assert len(cell_ms_df) == 15, f"E07_MultiSeed_Cell_Summary.csv must have 15 rows, got {len(cell_ms_df)}"
    assert len(seed_df) == 25, f"E07_Seed_Wise_Detail.csv must have 25 rows, got {len(seed_df)}"
    print("[Row-count assertions] PASS: 5 model summaries, 15 cell summaries, 25 seed-wise rows.")


## 14. Final Output Verification
## 15. Completion Summary


In [ ]:
# ==============================================================================
# SECTION 14-15: OUTPUT VERIFICATION AND COMPLETION SUMMARY
# ==============================================================================
n_completed = sum(1 for st in run_status.values() if st in ("COMPLETED", "SKIPPED_COMPLETE"))
failed = [(m, s, st) for (m, s), st in run_status.items() if st not in ("COMPLETED", "SKIPPED_COMPLETE")]

report = []
report.append("=" * 80)
report.append("E07 RUN REPORT")
report.append("=" * 80)
report.append(f"Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}")
report.append(f"RUN_MODE: {RUN_MODE}")
report.append(f"Device: {DEVICE}")
report.append(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
report.append(f"Total runs planned = 25")
report.append(f"Total runs completed or validated-skip = {n_completed}")
report.append(f"Failed runs = {len(failed)}")
report.append(f"Wall time (this session loop): {wall_total:.1f}s")
report.append(f"Reconciliation: {RECON_VERDICT}")
report.append("")
report.append("Model-by-model / seed-by-seed status")
for model_name in EXECUTION_ORDER:
    report.append(f"  {model_name}")
    for seed in SEEDS:
        st = run_status.get((model_name, seed), "NOT_STARTED")
        report.append(f"    seed {seed}: {st}")
if failed:
    report.append("FAILED RUNS:")
    for item in failed:
        report.append(f"  {item}")
else:
    report.append("Failed runs: none in this session.")
report.append("Warnings:")
report.append("  - E07 does not claim statistical significance.")
report.append("  - E07 does not claim universal robustness.")
report.append("  - E07 is not an unseen-temperature experiment.")
report.append("  - QNN-GRU batch_size=16 is a pre-existing E05/E06 protocol deviation.")
report.append("  - TE-Q campaign weight_decay=0.01 (E01 original used 0.05).")
report.append("=" * 80)
report_text = "\n".join(report)
(REPORTS_DIR / "E07_Run_Report.txt").write_text(report_text, encoding="utf-8")
print(report_text)

qa = []
qa.append("=" * 80)
qa.append("E07 FINAL QA")
qa.append("=" * 80)
qa.append("[x] Exactly one notebook created")
qa.append("[x] Five models included")
qa.append("[x] Models sourced from baselineComparison.ipynb (TE-Q from src/models/teq_transformer.py)")
qa.append("[x] No architecture rewrite")
qa.append("[x] Seeds exactly [42,43,44,45,46]")
qa.append("[x] 25 runs supported")
qa.append("[x] CUDA detection")
qa.append("[x] Kaggle compatible (relative /kaggle paths; no Windows drive hard-codes)")
qa.append("[x] NASA split verified")
qa.append("[x] B0018 unseen cell")
qa.append("[x] B0032 unseen cell")
qa.append("[x] B0053 70/30 temporal split")
qa.append("[x] Training-only scaler")
qa.append("[x] No test leakage")
qa.append("[x] Checkpointing (best + latest; training-loss selection)")
qa.append("[x] Resume support")
qa.append("[x] Prediction saving")
qa.append("[x] Training history")
qa.append("[x] Provenance")
qa.append("[x] Multi-seed aggregation")
qa.append("[x] Metric reconciliation")
qa.append("[x] No publication figures")
qa.append("[x] No full local CPU training in DRY_RUN (1 epoch / 1 batch per run)")
qa.append("")
qa.append(f"Pre-flight: {preflight_status}")
qa.append(f"Reconciliation: {RECON_VERDICT}")
qa.append(f"RUN_MODE: {RUN_MODE}")
qa.append("=" * 80)
(REPORTS_DIR / "E07_Final_QA.txt").write_text("\n".join(qa), encoding="utf-8")

audit_txt = []
audit_txt.append("=" * 80)
audit_txt.append("E07 IMPLEMENTATION AUDIT")
audit_txt.append("=" * 80)
audit_txt.append("model | source notebook section/cell | source implementation | configuration | parameter count | E05/E06 reference | E07 changes")
audit_txt.append("-" * 80)
for model_name in EXECUTION_ORDER:
    info = MODELS_REGISTRY[model_name]
    audit_txt.append(f"Model: {model_name}")
    audit_txt.append(f"  family: {info['family']}")
    audit_txt.append(f"  source notebook section/cell: {info['source_notebook_cell']}")
    audit_txt.append(f"  source implementation: {info['source_implementation']}")
    audit_txt.append(f"  configuration: AdamW lr={info['lr']} wd={info['weight_decay']} batch={info['batch_size']} max_epochs={info['max_epochs']} patience={info['patience']}")
    audit_txt.append(f"  parameter count: {info['expected_params']}")
    audit_txt.append(f"  E05/E06 reference: {info['e05_e06_reference']}")
    audit_txt.append(f"  E07 changes: {info['e07_changes']}")
    audit_txt.append(f"  Reused unchanged?: YES")
    audit_txt.append("")
audit_txt.append("Cell 4 of baselineComparison.ipynb is copied in full so shared helpers")
audit_txt.append("(_build_qnn_layer, RevIN, PositionalEncoding, etc.) remain byte-faithful.")
audit_txt.append("LSTM/GRU/CNN1D/TCN/DLinear/QLSTM classes are present in that cell but are")
audit_txt.append("NOT registered and NOT trained in E07.")
audit_txt.append("=" * 80)
(REPORTS_DIR / "E07_Implementation_Audit.txt").write_text("\n".join(audit_txt), encoding="utf-8")

import zipfile as _zipfile
zip_out = Path("E07_MultiSeed_Robustness_Results.zip")
with _zipfile.ZipFile(zip_out, "w", _zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_ROOT.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, file_path.relative_to(OUTPUT_ROOT.parent))
print(f"[Archive] {zip_out.resolve()} ({zip_out.stat().st_size / 1e6:.2f} MB)")

print("\n" + "#" * 75)
print("15. COMPLETION SUMMARY")
print(f"RUN_MODE: {RUN_MODE}")
print(f"Device: {DEVICE}")
print(f"Runs completed or verified-skip: {n_completed} / 25")
print(f"Reconciliation: {RECON_VERDICT}")
if RUN_MODE == "FULL":
    print("E07 FULL RUN LOOP COMPLETE")
    print("Download E07_results/ from the Kaggle output pane.")
else:
    print("E07 DRY_RUN pipeline check complete (not a scientific result).")
print("#" * 75)
